# Liquidity Provider Feed Quality Assessment: Series Overview

This notebook marks the beginning of a two-part series dedicated to the quantitative evaluation of liquidity provider feed quality from the perspective of ***client trading safety***.

The overall objective of the series is to decompose execution risk into its fundamental components and analyze each of them using a rigorous stochastic framework.

The analysis is structured along two key dimensions:

- **Part I — Spread Dynamics**, focusing on how the bid-ask spread behaves, how it expands, and how persistent and frequent these expansions are;
- **Part II — Quote Timing and Latency**, focusing on the temporal structure of quote arrivals, including delays, clustering effects, and their role in the formation of price gaps.

Each part isolates a distinct mechanism through which a liquidity provider can impact client outcomes. Together, they provide a comprehensive view of feed quality and execution risk.

The present notebook corresponds to **Part I** and is devoted entirely to the statistical analysis of spread behavior.

# Part I. Evaluation of Liquidity Provider Quality via Spread Dynamics Metrics

This notebook is the first component of a broader quantitative framework for evaluating liquidity providers from the standpoint of client trading safety.

The central premise is that ***the quality of a liquidity provider cannot be reduced to a simple comparison of average bid-ask spreads***. For leveraged clients, the main source of execution risk often arises during transient but severe microstructure deteriorations, when spreads widen materially, remain elevated for non-negligible time intervals, or recur with high frequency during specific trading sessions. Such spread distortions may increase transaction costs, worsen fill quality, accelerate margin depletion, and under stressed conditions contribute to stop-out events.

Accordingly, the purpose of this notebook is to analyze incoming quote streams with respect to four key characteristics of ***spread risk***:

1. spread expansion amplitude;
2. persistence of widened-spread states;
3. frequency of spread dislocations;
4. dependence of these effects on the trading session.

To capture these dynamics, we model the spread as a ***latent stochastic process with mean reversion***. The mathematical framework is based on the ***Ornstein-Uhlenbeck process and Kalman filtering***. This choice is justified both economically and statistically. From an economic standpoint, spread shocks are usually not permanent and tend to revert toward a session-specific equilibrium once temporary liquidity stress dissipates. From a statistical standpoint, observed spreads contain both a persistent latent component and transitory quote noise, making state-space modeling a natural approach.

For empirical illustration, we consider three anonymous liquidity providers and evaluate the quality of their spread formation. The goal is not merely to identify the narrowest provider on average, but to assess which provider offers the most stable and safest quoting environment for clients across different market regimes.

This notebook represents only ***the first stage*** of the full analysis. In subsequent notebooks, ***we will further investigate quote-update latency, measure the extent to which delayed quotes create executable price gaps***, and study whether such gaps can propagate into adverse client outcomes, including forced liquidation and stop-out risk.

Hence, the present part is devoted to the foundational layer of liquidity-provider diagnostics: ***the statistical structure of spread behavior***.

## Spread Widening evaluation:

The purpose of this analysis is to study and ***compare*** the quoted bid-ask ***spreads*** of ***several liquidity providers*** for the gold market. The focus is on understanding how spreads evolve over time, how they differ across providers, and whether their dynamics can be described by a ***structured stochastic model***.

From an economic perspective, **the spread is a key component of transaction costs. A tighter spread reduces execution costs for clients, while a wider spread reflects either increased uncertainty, lower liquidity, higher inventory risk, or a more conservative quoting policy by the provider.**

In this notebook, the empirical workflow combines:

1. loading and harmonizing quote data from several liquidity providers;
2. cleaning and preprocessing high-frequency observations;
3. aligning the data over a common time interval;
4. resampling quotes onto a regular time grid;
5. enriching the data with trading-session information;
6. estimating a session-dependent mean-reverting latent spread model.

The implementation uses numerical optimization, robust statistics, and state-space filtering tools in order to obtain economically interpretable parameters for each provider.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Tuple, Optional

from scipy.optimize import minimize
from tqdm.auto import tqdm

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 260)

## Data Sources and Input Structure

The raw data consist of quote streams from three liquidity providers:

\
$$
\text{LP}_1,\quad \text{LP}_2,\quad \text{LP}_3.
$$

\
Each dataset contains timestamped bid and ask quotes for gold. For each observation time $ t_i $, the provider publishes:

\
$
\text{Bid}_{t_i}, \qquad \text{Ask}_{t_i}.
$

\
The quoted spread is the difference between the ask and bid prices:

\
$
\text{Spread}_{t_i}^{\text{raw}} = \text{Ask}_{t_i} - \text{Bid}_{t_i}.
$

\
The midpoint quote is defined as

\
$
\text{Mid}_{t_i} = \frac{\text{Bid}_{t_i} + \text{Ask}_{t_i}}{2}.
$

\
At this stage, the objective is purely organizational: to load each liquidity provider into a unified tabular format and attach a provider label, so that all subsequent steps can be performed consistently across sources.

In [ ]:
PATH_LP1 = r"/content/sample_data/GOLD_Liquidity#1.csv"
PATH_LP2 = r"/content/sample_data/GOLD_Liquidity#2.csv"
PATH_LP3 = r"/content/sample_data/GOLD_Liquidity#3.csv"

## Standardization of Raw Quote Files

The function `load_liquidity_csv` converts each original file into a standardized data frame. The economic role of this step is to ensure comparability across providers by imposing a common schema.

The standardized fields are:

- $ \text{Date} $: timestamp of the quote,
- $ \text{Bid} $: quoted bid price,
- $ \text{Ask} $: quoted ask price,
- $ \text{Broker} $: identifier of the liquidity provider.

\
Formally, for each provider $ j \in \{1,2,3\} $, the imported dataset can be viewed as a sequence

\
$
\left\{ \left(t_i^{(j)},\, \text{Bid}_i^{(j)},\, \text{Ask}_i^{(j)}\right) \right\}_{i=1}^{N_j}.
$

\
This representation will later allow us to compare the statistical properties of quoted spreads across providers on a common basis.

In [ ]:
def load_liquidity_csv(path: str, broker_name: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        skiprows=1,
        sep=";",
        encoding="utf-16"
    ).copy()

    df.rename(columns={"Дата": "Date", "Бид": "Bid", "Аск": "Ask"}, inplace=True)

    if "Источник данных" in df.columns:
        df.drop(columns=["Источник данных"], inplace=True)

    df["Broker"] = broker_name
    return df


raw_feeds = {
    "Liquidity Provider #1": load_liquidity_csv(PATH_LP1, "Liquidity Provider #1"),
    "Liquidity Provider #2": load_liquidity_csv(PATH_LP2, "Liquidity Provider #2"),
    "Liquidity Provider #3": load_liquidity_csv(PATH_LP3, "Liquidity Provider #3"),
}

for name, df in raw_feeds.items():
    print(name, df.shape)
    display(df.head())



The object `raw_feeds` stores the quote history of each liquidity provider in a dictionary structure. Denoting the set of providers by

\
$
\mathcal{J} = \{1,2,3\},
$

the raw input can be written abstractly as

\
$
\mathcal{D}^{(j)} = \left\{ \left(t_i^{(j)}, \text{Bid}_i^{(j)}, \text{Ask}_i^{(j)}\right) \right\}_{i=1}^{N_j}, \qquad j \in \mathcal{J}.
$

\
At this point, the datasets are still raw in the sense that they may contain missing values, invalid prices, duplicated quotes, irregular arrival times, and potentially inconsistent spreads.



Before building any statistical model, it is necessary to inspect the raw feeds. This step serves two purposes.

First, it verifies that the data have been loaded correctly and that the column names and dimensions are consistent.

Second, it provides an initial sense of market microstructure heterogeneity across providers. In high-frequency data, even small inconsistencies may materially affect spread estimates, particularly when the model later relies on state-space filtering and session-level parameter estimation.

The main empirical questions at this stage are:

- How many observations are available for each provider?
- Are the timestamps parsed correctly?
- Are bid and ask quotes numerically valid?
- Do the feeds appear comparable in structure?



The preprocessing function transforms each raw quote stream into a clean and model-ready time series.


Each timestamp is converted into a valid datetime object, while bid and ask quotes are converted into numeric values.

Observations are excluded if any of the following conditions fail:

$
\text{Bid}_t > 0, \qquad \text{Ask}_t > 0, \qquad \text{Ask}_t \geq \text{Bid}_t.
$

The last condition is economically essential: the ask price must not be below the bid price, otherwise the quote would imply a negative spread, which is inconsistent with standard quoting conventions.

The observations are sorted chronologically and duplicate quote entries are removed. This ensures that the time series is properly ordered:

$
t_1 < t_2 < \cdots < t_N.
$


For each valid observation $ t_i $, the midpoint is computed as
$
M_{t_i} = \frac{\text{Bid}_{t_i} + \text{Ask}_{t_i}}{2},
$
and the raw spread as
$
S_{t_i}^{\text{raw}} = \text{Ask}_{t_i} - \text{Bid}_{t_i}.
$

\
The spread is also expressed in pips according to
$
S_{t_i}^{\text{pips}} = 100 \cdot \left(\text{Ask}_{t_i} - \text{Bid}_{t_i}\right).
$


\
Because quotes arrive asynchronously, the elapsed time between consecutive observations is computed:

\
$
\Delta t_i = t_i - t_{i-1}, \qquad i \geq 2.
$

\
In the implementation, this is measured in seconds and truncated below at zero:

\
$
\Delta t_i^{\text{sec}} = \max\left(0,\; (t_i - t_{i-1})_{\text{seconds}}\right).
$

\
This time increment will later play a crucial role in the continuous-time mean-reversion model, because the transition dynamics depend explicitly on the duration between observations.


After preprocessing, each provider $ j $ is represented by a cleaned time series:

$
\mathcal{X}^{(j)} = \left\{ \left(t_i^{(j)}, \text{Bid}_i^{(j)}, \text{Ask}_i^{(j)}, M_i^{(j)}, S_i^{(j)}\right) \right\}_{i=1}^{n_j},
$

where

$
M_i^{(j)} = \frac{\text{Bid}_i^{(j)} + \text{Ask}_i^{(j)}}{2},
\qquad
S_i^{(j)} = 100\left(\text{Ask}_i^{(j)} - \text{Bid}_i^{(j)}\right).
$

These cleaned feeds form the empirical basis for the later comparison of spread dynamics across liquidity providers.

In [ ]:
def preprocess_feed(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["Date"] = pd.to_datetime(out["Date"], format="%Y.%m.%d %H:%M:%S.%f", errors="coerce")
    out["Bid"] = pd.to_numeric(out["Bid"], errors="coerce")
    out["Ask"] = pd.to_numeric(out["Ask"], errors="coerce")

    out = out.dropna(subset=["Date", "Bid", "Ask"]).copy()
    out = out[(out["Bid"] > 0) & (out["Ask"] > 0)].copy()
    out = out[out["Ask"] >= out["Bid"]].copy()

    out = out.sort_values("Date").drop_duplicates(subset=["Date", "Bid", "Ask"]).reset_index(drop=True)

    out["Mid"] = (out["Bid"] + out["Ask"]) / 2.0
    out["Spread_pips"] = (out["Ask"] - out["Bid"]) * 100.0
    out["Spread_raw"] = out["Ask"] - out["Bid"]

    out["dt_sec"] = out["Date"].diff().dt.total_seconds().fillna(0.0).clip(lower=0.0)
    out["tick_id"] = np.arange(len(out))

    return out


feeds = {name: preprocess_feed(df) for name, df in raw_feeds.items()}

for name, df in feeds.items():
    print(name, df.shape, df["Date"].min(), df["Date"].max())
    display(df[["Date", "Bid", "Ask", "Mid", "Spread_pips"]].head())



After preprocessing, the cleaned data are inspected again to verify that the transformation has produced economically meaningful spread series.

\
At this stage, the key variables of interest are:


- $ \text{Date} $,
- $ \text{Bid} $,
- $ \text{Ask} $,
- $ \text{Mid} $,
- $ \text{Spread}_{\text{pips}} $.

\
This inspection confirms that the observed spread process is well-defined and ready for synchronization across providers.



A direct comparison of providers is ***only meaningful over the time interval during which all providers are simultaneously observed***.

\
Let the first and last timestamps of provider $ j $ be
$
T_{\min}^{(j)} = \min_i t_i^{(j)},
\qquad T_{\max}^{(j)} = \max_i t_i^{(j)}.
$

\
The common sample window is then defined by
$
T_{\text{start}} = \max_j T_{\min}^{(j)},
\qquad
T_{\text{end}} = \min_j T_{\max}^{(j)}.
$

\
This ensures that all providers are compared on the same time support:

\
$
[T_{\text{start}},\, T_{\text{end}}].
$

\
***Economically, this avoids drawing conclusions from non-overlapping market conditions***. For example, a provider should not appear better merely because its sample excludes a volatile or illiquid market episode captured by another provider.

In [ ]:
def get_common_time_window(feeds: Dict[str, pd.DataFrame]) -> Tuple[pd.Timestamp, pd.Timestamp]:
    start = max(df["Date"].min() for df in feeds.values())
    end = min(df["Date"].max() for df in feeds.values())
    return start, end


common_start, common_end = get_common_time_window(feeds)
print("Common start:", common_start)
print("Common end  :", common_end)


def trim_to_common_window(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    out = df[(df["Date"] >= start) & (df["Date"] <= end)].copy().reset_index(drop=True)
    out["dt_sec"] = out["Date"].diff().dt.total_seconds().fillna(0.0).clip(lower=0.0)
    out["tick_id"] = np.arange(len(out))
    return out


feeds_common = {name: trim_to_common_window(df, common_start, common_end) for name, df in feeds.items()}



Two regular grids are introduced:

- a coarser grid for model estimation,
- a finer grid for visualization.

\
Denote the grid step by $ \delta $. In the notebook:

- the model grid uses $ \delta = 5 $ seconds,
- the visualization grid uses $ \delta = 1 $ second.

\
***The motivation is practical***. Model estimation on a coarser grid reduces computational burden and limits excessive sensitivity to microsecond-level quote noise, while the finer grid preserves a more detailed visual representation of spread movements.


The original quote arrivals are irregular in time. To make providers comparable and to simplify state-space estimation, the quotes are resampled onto a regular time grid.

For each grid interval, the last available quote is retained. If the grid times are denoted by
$
\tau_1, \tau_2, \ldots, \tau_m,
$
then the resampled quote at time $ \tau_k $ is the most recent observed quote within the corresponding bin.

\
If no new quote appears exactly at a given grid point, the most recent valid quote is carried forward. This implements the idea that the last available quote remains active until updated.

\
However, to avoid using excessively stale quotes, forward filling is limited by a maximum quote age. Let $ a_k $ denote the age of the last observed quote at grid time $ \tau_k $. Then the carried-forward quote is accepted only if
$
a_k \leq A_{\max},
$
where $ A_{\max} $ is the forward-fill limit in seconds.

\
This is economically important because a very old quote may no longer represent executable liquidity. Without such a restriction, one might create artificial flat segments in the spread series.

After resampling, the midpoint and spread are recomputed:

\
$
M_{\tau_k} = \frac{\text{Bid}_{\tau_k} + \text{Ask}_{\tau_k}}{2},
$

\
$
S_{\tau_k}^{\text{raw}} = \text{Ask}_{\tau_k} - \text{Bid}_{\tau_k},
$

\
$
S_{\tau_k}^{\text{pips}} = 100 \cdot \left(\text{Ask}_{\tau_k} - \text{Bid}_{\tau_k}\right).
$

\
The elapsed time between adjacent grid points is also stored as
$
\Delta \tau_k = \tau_k - \tau_{k-1}.
$

Although the grid is regular in construction, some gaps may remain after stale-quote filtering, so explicit time increments are still useful.

In [ ]:
RESAMPLE_FREQ_MODEL = "5S"
RESAMPLE_FREQ_VIZ = "1S"

def resample_quotes_to_grid(df: pd.DataFrame, freq: str = "5S", ffill_limit_sec: Optional[int] = 30) -> pd.DataFrame:
    """
    last quote per bin + ffill
    optionally limit ffill by time gap (seconds), to avoid "stale quote plateaus"
    """
    x = df.copy().sort_values("Date")
    broker = x["Broker"].iloc[0] if "Broker" in x.columns and len(x) else "Unknown"

    # last in bin
    rs = x.set_index("Date")[["Bid", "Ask"]].resample(freq).last()

    if ffill_limit_sec is None:
        rs = rs.ffill()
    else:
        rs_ff = rs.ffill()
        last_obs_time = rs.notna().all(axis=1)
        last_obs_time = rs.index.to_series().where(last_obs_time).ffill()
        age_sec = (rs.index.to_series() - last_obs_time).dt.total_seconds()
        rs_ff = rs_ff.where(age_sec <= float(ffill_limit_sec))
        rs = rs_ff

    rs = rs.dropna().copy()

    out = rs.reset_index()
    out["Broker"] = broker

    out["Mid"] = (out["Bid"] + out["Ask"]) / 2.0
    out["Spread_pips"] = (out["Ask"] - out["Bid"]) * 100.0
    out["Spread_raw"] = out["Ask"] - out["Bid"]

    out["dt_sec"] = out["Date"].diff().dt.total_seconds().fillna(0.0).clip(lower=0.0)
    out["tick_id"] = np.arange(len(out))
    return out


feeds_grid_model = {name: resample_quotes_to_grid(df, RESAMPLE_FREQ_MODEL, ffill_limit_sec=60) for name, df in feeds_common.items()}
feeds_grid_viz   = {name: resample_quotes_to_grid(df, RESAMPLE_FREQ_VIZ,   ffill_limit_sec=20) for name, df in feeds_common.items()}

for name in feeds_grid_model:
    print(name, "model:", len(feeds_grid_model[name]), "viz:", len(feeds_grid_viz[name]))


Two resampled datasets are built for each provider:


1. **Model sample**: a regularized series used for parameter estimation;
2. **Visualization sample**: a finer series used for plotting and descriptive analysis.

\
Let
$
\mathcal{X}_{\text{model}}^{(j)}
\quad \text{and} \quad
\mathcal{X}_{\text{viz}}^{(j)}
$
denote the corresponding resampled series for provider $ j $.

\
This separation is useful because the statistical model does not require the same resolution as the visual presentation. Estimation benefits from a more stable and less noisy grid, while visualization benefits from higher temporal resolution.


The spread process is expected to ***differ across major global trading sessions***. To capture this heterogeneity, each timestamp is assigned to one of four sessions:

\
$$
\text{Asia}, \qquad \text{Europe}, \qquad \text{US}, \qquad \text{Rollover}.
$$

\
This partitions the day into regimes with distinct liquidity conditions. In economic terms:

- the **Asian session** may feature thinner liquidity for some instruments,
- the **European session** often corresponds to deeper participation,
- the **US session** may combine high activity with macroeconomic releases,
- the **rollover period** can show temporary widening due to lower market depth and operational adjustments.

Thus, session assignment introduces a regime structure into the spread dynamics.

In [ ]:
def assign_session(ts: pd.Timestamp) -> str:
    h = ts.hour
    if 1 <= h < 10:
        return "Asia"
    elif 10 <= h < 16:
        return "Europe"
    elif 16 <= h < 24:
        return "US"
    else:
        return "Rollover"


def add_calendar_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["trade_date"] = out["Date"].dt.date.astype(str)
    out["hour"] = out["Date"].dt.hour
    out["session"] = out["Date"].apply(assign_session)
    return out


SESSION_ORDER = ["Asia", "Europe", "US", "Rollover"]
SESSION_TO_INT = {s: i for i, s in enumerate(SESSION_ORDER)}
INT_TO_SESSION = {i: s for s, i in SESSION_TO_INT.items()}


def add_session_codes(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["session_code"] = out["session"].map(SESSION_TO_INT).astype(int)
    return out


feeds_grid_model = {name: add_session_codes(add_calendar_columns(df)) for name, df in feeds_grid_model.items()}
feeds_grid_viz   = {name: add_session_codes(add_calendar_columns(df)) for name, df in feeds_grid_viz.items()}


Each observation is enriched with calendar information:


- trading date,
- hour of day,
- trading session.

\
Formally, for each timestamp $ t_i $, we construct a mapping

\
$
t_i \mapsto \left(\text{date}_i,\; \text{hour}_i,\; \text{session}_i\right)
$

\
These calendar variables are not merely descriptive. They provide the regime labels that will later determine the session-specific long-run mean, volatility scale, and measurement noise in the stochastic spread model.


For estimation purposes, each categorical session label is mapped into an integer code:

\
$
\text{Asia} \mapsto 0,\qquad
\text{Europe} \mapsto 1,\qquad
\text{US} \mapsto 2,\qquad
\text{Rollover} \mapsto 3.
$

\
Denote the session code at time $ t_i $ by $ s_i \in \{0,1,2,3\} $.

\
This encoding allows the model parameters to be written compactly as session-indexed vectors, such as

\
$
\boldsymbol{\mu} = (\mu_0,\mu_1,\mu_2,\mu_3),
\qquad
\boldsymbol{\sigma} = (\sigma_0,\sigma_1,\sigma_2,\sigma_3),
\qquad
\mathbf{R} = (R_0,R_1,R_2,R_3).
$


After assigning each observation to a trading session, the series is augmented with a numerical regime indicator $ s_i $.

The session code will determine which parameter values apply at each observation. Thus, the model is not fully homogeneous through time; instead, it is a **session-dependent latent mean-reverting model**.

This means that the spread is allowed to revert toward different normal levels in different parts of the trading day.


A robust estimator of scale is required because high-frequency spread data may contain jumps, outliers, and transient distortions.

\
For a sample $ x_1,\dots,x_n $, let the median be

\
$
m = \operatorname{median}(x_1,\dots,x_n).
$

\
The median absolute deviation is defined as

\
$
\operatorname{MAD}(x) = \operatorname{median}\left( |x_i - m| \right).
$

\
To convert this into a Gaussian-consistent estimator of the standard deviation, the notebook uses

\
$
\hat{\sigma}_{\text{MAD}} = 1.4826 \cdot \operatorname{MAD}(x).
$

\
This estimator is much less sensitive to extreme observations than the classical standard deviation. In a market microstructure setting, this is especially useful because spread spikes may be rare but very large.

In [ ]:
def robust_mad_sigma(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 1.4826 * mad


def seconds_to_steps(df: pd.DataFrame, window_sec: int) -> int:
    dt = df["dt_sec"].values
    dt = dt[(dt > 0) & np.isfinite(dt)]
    step = float(np.median(dt)) if len(dt) else 1.0
    return int(max(1, round(window_sec / max(step, 1e-9))))


def rolling_robust_sigma_by_seconds(series: pd.Series, df_ref: pd.DataFrame, window_sec: int) -> pd.Series:
    steps = seconds_to_steps(df_ref, window_sec)
    min_periods = max(10, steps // 5)

    def _f(x):
        return robust_mad_sigma(np.asarray(x))

    return series.rolling(steps, min_periods=min_periods).apply(_f, raw=False)


Some descriptive statistics are defined over a time horizon measured in seconds rather than in a fixed number of observations.

If the median sampling interval is

\
$
\tilde{\Delta t} = \operatorname{median}\{\Delta t_i : \Delta t_i > 0\},
$

\
then a window of $ W $ seconds is converted into an approximate number of steps by

\
$
n_W = \max\left(1,\; \operatorname{round}\left(\frac{W}{\tilde{\Delta t}}\right)\right).
$

\:
This translation is needed because the data may be irregular or partially filtered, so the relationship between calendar time and observation count is not trivial.


To measure local variability of the spread over time, the notebook computes a rolling robust scale estimator.

Given a spread series $ \{S_t\} $, a time window of $ W $ seconds is mapped to $ n_W $ observations, and the rolling statistic is

\
$
\hat{\sigma}_{t}^{(W)} =
1.4826 \cdot
\operatorname{median}
\left(
\left| S_{t-k} - \operatorname{median}(S_{t-n_W+1},\dots,S_t) \right|
\right),
$

\
computed over the local window.

This quantity provides a robust measure of local spread instability, which may differ sharply across providers and sessions.


The model parameters are collected in the object `SessionwiseMixedParams`. For each provider, the parameter set is

\
$$
\Theta =
\left(
\boldsymbol{\mu},
\; h,
\; \boldsymbol{\sigma},
\; \mathbf{R}
\right),
$$

where

$
\boldsymbol{\mu} = (\mu_0,\mu_1,\mu_2,\mu_3),
$

$
h > 0
$
is the half-life of mean reversion in seconds,

$
\boldsymbol{\sigma} = (\sigma_0,\sigma_1,\sigma_2,\sigma_3)
$
are the session-specific stationary standard deviations of the latent process, and

$
\mathbf{R} = (R_0,R_1,R_2,R_3)
$
are the session-specific observation noise variances.

\
The mean-reversion speed $ \kappa $ is not estimated independently, but derived from the half-life:

$
\kappa = \frac{\ln 2}{h}.
$

\
This parameterization is economically intuitive because the half-life directly answers the question:

> How long does it take for a deviation of the spread from its equilibrium level to decay by one half?

In [ ]:
@dataclass
class SessionwiseMixedParams:
    mu: np.ndarray            # (4,)
    half_life: float          # scalar (seconds)
    stationary_sd: np.ndarray # (4,)
    R: np.ndarray             # (4,)

    @property
    def kappa(self) -> float:
        return float(np.log(2.0) / max(self.half_life, 1e-6))


def unpack_theta(theta: np.ndarray) -> SessionwiseMixedParams:
    """
    theta = [mu_0..mu_3, log_half_life, log_sd_0..log_sd_3, log_R_0..log_R_3]  => 13 params
    """
    theta = np.asarray(theta, dtype=float).ravel()
    if theta.size != 13:
        raise ValueError(f"theta must have 13 params: mu4 + logHL + logsd4 + logR4, got {theta.size}")

    mu = theta[0:4]
    half_life = float(np.exp(theta[4]))
    stationary_sd = np.exp(theta[5:9])
    R = np.exp(theta[9:13])
    return SessionwiseMixedParams(mu=mu, half_life=half_life, stationary_sd=stationary_sd, R=R)


Numerical optimization is easier when all parameters live on the real line. Therefore, the parameter vector is represented as

\
$$
\theta =
\left(
\mu_0,\mu_1,\mu_2,\mu_3,\;
\log h,\;
\log \sigma_0,\log \sigma_1,\log \sigma_2,\log \sigma_3,\;
\log R_0,\log R_1,\log R_2,\log R_3
\right).
$$

\
The transformation back to economically meaningful parameters is

$
h = e^{\theta_4},
\qquad
\sigma_s = e^{\theta_{5+s}},
\qquad
R_s = e^{\theta_{9+s}},
\qquad s=0,1,2,3.
$

\
This ensures positivity:

$
h > 0, \qquad \sigma_s > 0, \qquad R_s > 0.
$

Such a log-parameterization is standard in state-space estimation because it avoids invalid negative variance estimates during optimization.


The half-life parameter is restricted to a plausible interval:

\
$
h_{\min} \leq h \leq h_{\max},
$

\
where in the implementation

\
$
h_{\min} = 3 \text{ seconds},
\qquad
h_{\max} = 6 \text{ hours}.
$

\
This restriction prevents the optimizer from selecting economically implausible extremes:

- an almost instantaneous reversion that would imply the spread snaps back unrealistically fast;
- an excessively slow reversion that would effectively destroy the intended mean-reverting interpretation.


Good initialization is important for stable maximum-likelihood estimation.



\
For each session $ s $, the initial long-run mean is set equal to the median spread within that session:

\
$
\mu_s^{(0)} = \operatorname{median}\left\{ S_i : s_i = s \right\}.
$

\
If a session has too few observations, the global median is used instead.


\
Let the spread series be $ y_i = S_i $. The first differences are
$
\Delta y_i = y_i - y_{i-1}.
$

\
A robust scale estimate of $ \Delta y_i $ provides an initial measure of local variation:

\
$
\hat{\sigma}_{\Delta y} = 1.4826 \cdot \operatorname{MAD}(\Delta y).
$


\
If the lag-one correlation is approximately $ \rho $, and the typical time step is $ \tilde{\Delta t} $, then under an ***Ornstein-Uhlenbeck***-type logic
$
\rho \approx e^{-\kappa \tilde{\Delta t}}.
$

\
Hence the initial mean-reversion speed is
$
\kappa^{(0)} = -\frac{\ln \rho}{\tilde{\Delta t}},
$
and the corresponding half-life is
$
h^{(0)} = \frac{\ln 2}{\kappa^{(0)}}.
$


\
The initial stationary standard deviation is set to a positive scale estimate:
$
\sigma_s^{(0)} = \max(\hat{\sigma}_{\Delta y},\, 10^{-4}).
$

\
The initial measurement noise variance is chosen as
$
R_s^{(0)} = \max\left(\left(\frac{\hat{\sigma}_{\Delta y}}{\sqrt{2}}\right)^2,\; 10^{-6}\right).
$

\
These starting values are not yet economically final estimates, but they place the optimizer in a plausible region of the parameter space.

In [ ]:
HALF_LIFE_MIN_SEC = 3.0
HALF_LIFE_MAX_SEC = 6.0 * 3600.0  # 6 hours

def make_initial_theta(df: pd.DataFrame, spread_col: str = "Spread_pips") -> np.ndarray:
    y = df[spread_col].values.astype(float)
    sc = df["session_code"].values.astype(int)
    dt = df["dt_sec"].values.astype(float)

    # mu_s init = medians by session
    mu = np.zeros(4, dtype=float)
    for s in range(4):
        ys = y[sc == s]
        mu[s] = float(np.median(ys)) if len(ys) else float(np.median(y))

    # scale init
    dy = np.diff(y)
    sig_diff = robust_mad_sigma(dy)
    if not np.isfinite(sig_diff) or sig_diff <= 1e-8:
        sig_diff = max(np.std(dy), 1e-6)

    # half-life init from lag-1 corr
    y0, y1 = y[:-1], y[1:]
    if len(y0) > 10 and np.std(y0) > 0 and np.std(y1) > 0:
        rho = float(np.clip(np.corrcoef(y0, y1)[0, 1], 1e-4, 0.9999))
    else:
        rho = 0.95

    positive_dt = dt[dt > 0]
    dt_med = max(float(np.median(positive_dt)) if len(positive_dt) else 1.0, 1e-6)
    kappa0 = max(-np.log(rho) / dt_med, 1e-4)
    half_life0 = float(np.log(2.0) / kappa0)
    half_life0 = float(np.clip(half_life0, HALF_LIFE_MIN_SEC, HALF_LIFE_MAX_SEC))

    stationary_sd0 = max(sig_diff, 1e-4)
    R0 = max((sig_diff / np.sqrt(2.0)) ** 2, 1e-6)

    theta0 = np.concatenate([
        mu,
        np.array([np.log(half_life0)]),
        np.log(np.full(4, stationary_sd0)),
        np.log(np.full(4, R0))
    ]).astype(float)

    return theta0

## Stochastic-Process Foundation of the ***Ornstein-Uhlenbeck Spread Model***

We now provide the full stochastic-process foundation for the latent spread specification. The key modeling assumption is that the underlying efficient spread is not a pure random walk and not an independent sequence of shocks. Instead, it is a continuous-time Markov diffusion with a restoring force toward a regime-dependent equilibrium level. The canonical Gaussian diffusion with these properties is the ***Ornstein-Uhlenbeck process***.

---

### 1. Economic motivation for a mean-reverting diffusion

Let $ X_t $ denote the latent efficient spread at time $ t $. We postulate that deviations of $ X_t $ from its equilibrium level are temporary. This is economically natural in dealer and liquidity-provider settings:

- if the spread becomes unusually wide, competition and normalization of inventory pressure tend to compress it back;
- if the spread becomes unusually narrow, inventory risk, adverse selection concerns, and reduced compensation for market making tend to push it upward again.

\
Hence the drift should be directed toward a central level $ \mu $, and its sign should oppose the deviation $ X_t - \mu $. The simplest linear specification is

\
$$
b(x) = \kappa(\mu - x),
\qquad \kappa > 0.
$$

\
To model random liquidity shocks, asynchronous quote revisions, and other market frictions, we add a diffusion term. Thus the latent spread is modeled by the stochastic differential equation

\
$$
\boxed{
dX_t = \kappa(\mu - X_t)\,dt + \sigma\,dW_t,
}
$$

\
where $ \sigma > 0 $ and $ \{W_t\}_{t\ge 0} $ is a standard Brownian motion.

\
This is the ***Ornstein-Uhlenbeck process***.

---

### 2. Existence, uniqueness, and Markov structure

The coefficients of the stochastic differential equation are

\
$$
b(x) = \kappa(\mu-x),
\qquad
a(x) = \sigma.
$$

\
The drift $ b(x) $ is globally Lipschitz and of linear growth, and the diffusion coefficient is constant. Therefore the stochastic differential equation admits a unique strong solution for every initial condition $ X_0 \in L^2 $. In particular, for every deterministic or square-integrable random initial value $ X_0 $, there exists a unique adapted continuous process $ \{X_t\}_{t\ge 0} $ satisfying

\
$$
X_t = X_0 + \int_0^t \kappa(\mu - X_s)\,ds + \sigma W_t.
$$

\
Because the stochastic differential equation is time-homogeneous and driven by Brownian motion, $ X_t $ is a time-homogeneous Markov process. Its future conditional law depends on the present state only:

\
$$
\mathbb{P}(X_t \in A \mid \mathcal{F}_s) = \mathbb{P}(X_t \in A \mid X_s),
\qquad t \ge s.
$$

\
This Markov property is exactly what later makes ***recursive filtering possible.***

---

### 3. Explicit solution by integrating factor

Starting from

\
$$
dX_t = \kappa(\mu - X_t)\,dt + \sigma\,dW_t,
$$

\
rewrite it as

\
$$
dX_t + \kappa X_t\,dt = \kappa\mu\,dt + \sigma\,dW_t.
$$

\
Multiply both sides by the integrating factor $ e^{\kappa t} $. Using Itô's product rule,

\
$$
d\bigl(e^{\kappa t}X_t\bigr)
=
e^{\kappa t}dX_t + \kappa e^{\kappa t}X_t\,dt.
$$

\
Therefore

\
$$
d\bigl(e^{\kappa t}X_t\bigr)
=
\kappa\mu e^{\kappa t}\,dt + \sigma e^{\kappa t}\,dW_t.
$$

\
Integrating from $ s $ to $ t $,

\
$$
e^{\kappa t}X_t - e^{\kappa s}X_s
=
\kappa\mu \int_s^t e^{\kappa u}\,du
+
\sigma\int_s^t e^{\kappa u}\,dW_u.
$$

\
Hence

\
$$
X_t
=
e^{-\kappa(t-s)}X_s
+
\mu\bigl(1-e^{-\kappa(t-s)}\bigr)
+
\sigma\int_s^t e^{-\kappa(t-u)}\,dW_u.
$$

\
So the exact solution is

\
$$
\boxed{
X_t
=
\mu + e^{-\kappa(t-s)}(X_s-\mu)
+
\sigma\int_s^t e^{-\kappa(t-u)}\,dW_u.
}
$$

\
This representation already reveals all key properties:

- exponential decay of memory;
- Gaussian transition law;
- exact discretization on irregular time intervals.

---

### 4. Conditional mean, conditional variance, and Gaussian transition law

Conditionally on $ X_s $, the stochastic integral

\
$$
\int_s^t e^{-\kappa(t-u)}\,dW_u
$$

\
is centered Gaussian. Therefore $ X_t \mid X_s $ is Gaussian.

#### Conditional mean


$$
\mathbb{E}[X_t \mid X_s]
=
\mu + e^{-\kappa(t-s)}(X_s-\mu).
$$


#### Conditional variance


By Itô isometry,

\
$$
\operatorname{Var}(X_t \mid X_s)
=
\sigma^2 \int_s^t e^{-2\kappa(t-u)}\,du.
$$

\
Evaluating the integral,

\
$$
\operatorname{Var}(X_t \mid X_s)
=
\sigma^2 \frac{1-e^{-2\kappa(t-s)}}{2\kappa}.
$$

\
Therefore

\
$$
\boxed{
X_t \mid X_s
\sim
\mathcal{N}
\left(
\mu + e^{-\kappa(t-s)}(X_s-\mu),
\;
\frac{\sigma^2}{2\kappa}\bigl(1-e^{-2\kappa(t-s)}\bigr)
\right).
}
$$

\
If we define

\
$$
a(t-s) = e^{-\kappa(t-s)},
\qquad
v(t-s) = \frac{\sigma^2}{2\kappa}\bigl(1-a(t-s)^2\bigr),
$$

\
then

\
$$
X_t \mid X_s \sim \mathcal{N}\bigl(\mu + a(t-s)(X_s-\mu),\, v(t-s)\bigr).
$$

\
This exact Gaussian transition kernel is the probabilistic backbone of the ***Kalman state equation***.

---

### 5. Infinitesimal generator

The infinitesimal generator $ \mathcal{L} $ of the Ornstein-Uhlenbeck diffusion acts on sufficiently smooth test functions $ f \in C^2 $ as

\
$$
\boxed{
\mathcal{L}f(x)
=
\kappa(\mu-x)f'(x) + \frac{\sigma^2}{2}f''(x).
}
$$

\
This follows from Itô's formula. For any $ f \in C^2 $,

\
$$
df(X_t)
=
f'(X_t)\,dX_t + \frac12 f''(X_t)\,(dX_t)^2,
$$

\
so substituting the dynamics of $ X_t $,

\
$$
df(X_t)
=
\left[
\kappa(\mu-X_t)f'(X_t) + \frac{\sigma^2}{2}f''(X_t)
\right]dt
+
\sigma f'(X_t)\,dW_t.
$$

\
Hence

\
$$
f(X_t) - f(X_0) - \int_0^t \mathcal{L}f(X_s)\,ds
$$

\
is a ***martingale***.

\
The generator characterizes the local dynamics completely and is the analytical object behind the backward and forward Kolmogorov equations.

---

### 6. Kolmogorov backward equation and Markov semigroup

Define the Markov semigroup

\
$$
P_t f(x) = \mathbb{E}\bigl[f(X_t)\mid X_0=x\bigr].
$$

\
Then $ u(t,x)=P_t f(x) $ solves the backward Kolmogorov equation

\
$$
\frac{\partial u}{\partial t}(t,x)
=
\mathcal{L}u(t,x),
\qquad
u(0,x)=f(x),
$$

\
that is,

\
$$
\boxed{
\frac{\partial u}{\partial t}(t,x)
=
\kappa(\mu-x)\frac{\partial u}{\partial x}(t,x)
+
\frac{\sigma^2}{2}\frac{\partial^2 u}{\partial x^2}(t,x).
}
$$

\
Thus the ***Ornstein-Uhlenbeck semigroup*** is generated by a second-order elliptic operator with linear drift and constant diffusion.

---

### 7. Fokker-Planck equation and density evolution

If $ p(t,x) $ denotes the transition density or the time-$ t $ marginal density of $ X_t $, then $ p $ satisfies the forward Kolmogorov equation, also called the ***Fokker-Planck equation***:

\
$$
\frac{\partial p}{\partial t}(t,x)
=
-\frac{\partial}{\partial x}\bigl(\kappa(\mu-x)p(t,x)\bigr)
+
\frac{\sigma^2}{2}\frac{\partial^2 p}{\partial x^2}(t,x).
$$

\
Equivalently,

\
$$
\boxed{
\frac{\partial p}{\partial t}(t,x)
=
\kappa \frac{\partial}{\partial x}\bigl((x-\mu)p(t,x)\bigr)
+
\frac{\sigma^2}{2}\frac{\partial^2 p}{\partial x^2}(t,x).
}
$$

\
This partial differential equation describes the evolution of the full law of the latent spread, not just its moments.

---

### 8. Invariant distribution from the stationary Fokker-Planck equation

An invariant density $ \pi(x) $ must satisfy

\
$$
0
=
-\frac{d}{dx}\bigl(\kappa(\mu-x)\pi(x)\bigr)
+
\frac{\sigma^2}{2}\frac{d^2}{dx^2}\pi(x).
$$



Under the natural zero-probability-current condition, the stationary flux is zero:

\
$$
\kappa(\mu-x)\pi(x) - \frac{\sigma^2}{2}\pi'(x) = 0.
$$


Therefore

\
$$
\pi'(x)
=
\frac{2\kappa(\mu-x)}{\sigma^2}\pi(x).
$$


Dividing by $ \pi(x) $,

$$
\frac{\pi'(x)}{\pi(x)}
=
\frac{2\kappa(\mu-x)}{\sigma^2}.
$$

Integrating,

\
$$
\log \pi(x)
=
-\frac{\kappa}{\sigma^2}(x-\mu)^2 + C.
$$

Hence

\
$$
\pi(x)
=
C
\exp\left(
-\frac{\kappa}{\sigma^2}(x-\mu)^2
\right).
$$

\
Normalizing,

\
$$
\boxed{
\pi(x)
=
\sqrt{\frac{\kappa}{\pi\sigma^2}}
\exp\left(
-\frac{\kappa}{\sigma^2}(x-\mu)^2
\right).
}
$$

\
This is the density of a Gaussian distribution with mean $ \mu $ and variance $ \sigma^2/(2\kappa) $. Therefore

\
$$
\boxed{
X_t^{(\mathrm{stat})}
\sim
\mathcal{N}\left(\mu,\frac{\sigma^2}{2\kappa}\right).
}
$$

\
This derivation is important because it shows that the normal stationary law is not merely guessed but follows analytically from the ***Fokker-Planck equation***.

---

### 9. First and second moments

Using the explicit solution, the unconditional mean satisfies

$$
\mathbb{E}[X_t]
=
\mu + e^{-\kappa t}\bigl(\mathbb{E}[X_0]-\mu\bigr).
$$

Hence

$$
\boxed{
\mathbb{E}[X_t] \to \mu \quad \text{as } t\to\infty.
}
$$

For the variance, one obtains

$$
\operatorname{Var}(X_t)
=
e^{-2\kappa t}\operatorname{Var}(X_0)
+
\frac{\sigma^2}{2\kappa}\bigl(1-e^{-2\kappa t}\bigr).
$$

Thus

$$
\boxed{
\operatorname{Var}(X_t)\to \frac{\sigma^2}{2\kappa}
\quad \text{as } t\to\infty.
}
$$

If $ X_0 \sim \mathcal{N}(\mu,\sigma^2/(2\kappa)) $, then the process is strictly stationary.

---

### 10. Covariance and autocorrelation structure

Assume stationarity. Then for $ h \ge 0 $,

\
$$
X_{t+h}
=
\mu + e^{-\kappa h}(X_t-\mu)
+
\sigma\int_t^{t+h} e^{-\kappa(t+h-u)}\,dW_u.
$$

\
The stochastic integral over $ [t,t+h] $ is independent of $ X_t $, hence

\
$$
\operatorname{Cov}(X_t,X_{t+h})
=
e^{-\kappa h}\operatorname{Var}(X_t).
$$

\
Under stationarity,

\
$$
\operatorname{Var}(X_t) = \frac{\sigma^2}{2\kappa},
$$

\
so

\
$$
\boxed{
\operatorname{Cov}(X_t,X_{t+h})
=
\frac{\sigma^2}{2\kappa}e^{-\kappa h}.
}
$$

\
Therefore the autocorrelation function is

\
$$
\boxed{
\rho(h) = e^{-\kappa h}.
}
$$

\
This exponential covariance decay is exactly the feature later exploited when initializing the half-life from empirical lag-one autocorrelation.

---

### 11. Half-life of mean reversion

The conditional mean deviation from equilibrium evolves as

\
$$
\mathbb{E}[X_t-\mu \mid X_s] = e^{-\kappa(t-s)}(X_s-\mu).
$$

\
The half-life $ \tau_{1/2} $ is defined by

\
$$
e^{-\kappa \tau_{1/2}} = \frac12.
$$

\
Hence

\
$$
\boxed{
\tau_{1/2} = \frac{\ln 2}{\kappa}.
}
$$

\
Equivalently,

\
$$
\boxed{
\kappa = \frac{\ln 2}{\tau_{1/2}}.
}
$$

\
This is why the code parameterizes the model through a half-life: it is directly interpretable in economic terms as the time needed for half of a spread shock to dissipate.

---

### 12. Reparameterization through stationary variance

Instead of parameterizing the process directly by $ \sigma $, it is often more interpretable to use the stationary variance

\
$$
\omega^2 := \frac{\sigma^2}{2\kappa}.
$$

\
Then the conditional law becomes

\
$$
\boxed{
X_t \mid X_s
\sim
\mathcal{N}
\left(
\mu + e^{-\kappa(t-s)}(X_s-\mu),
\;
\omega^2\bigl(1-e^{-2\kappa(t-s)}\bigr)
\right).
}
$$

\
So the ***Ornstein-Uhlenbeck model*** may be parameterized equivalently by

\
$$
(\mu,\kappa,\sigma)
\qquad \text{or} \qquad
(\mu,\kappa,\omega).
$$

\
The code uses the latter idea sessionwise, where $ \omega $ is the session-specific stationary standard deviation.

---

### 13. Exact discretization on an irregular observation grid

Suppose the process is observed at times

\
$$
0 \le t_1 < t_2 < \cdots < t_n,
\qquad
\Delta t_k = t_k - t_{k-1}.
$$

\
Then the exact discrete-time transition is

\
$$
X_k
=
\mu + a_k(X_{k-1}-\mu) + \eta_k,
$$

\
where

\
$$
a_k = e^{-\kappa \Delta t_k},
$$

\
and

\
$$
\eta_k \sim \mathcal{N}(0,q_k),
\qquad
q_k = \omega^2(1-a_k^2).
$$

\
Equivalently, in original diffusion parameters,

\
$$
q_k = \frac{\sigma^2}{2\kappa}\bigl(1-e^{-2\kappa \Delta t_k}\bigr).
$$

\
This exact discretization is extremely important: the model does not rely on an Euler approximation. It preserves the exact Gaussian transition law even when the observation intervals are irregular.

---

### 14. Regime-switching mean level across sessions

Let $ s_k \in \{0,1,2,3\} $ denote the trading session of the $ k $-th observation. The equilibrium level and stationary volatility scale are allowed to depend on $ s_k $. Thus we define

\
$$
\mu_{s_k}, \qquad \omega_{s_k}, \qquad R_{s_k}.
$$

\
The transition becomes

\
$$
\boxed{
X_k
=
\mu_{s_k}
+
a_k\bigl(X_{k-1} - \mu_{s_{k-1}}\bigr)
+
\eta_k,
}
$$

\
where

\
$$
\eta_k \sim \mathcal{N}(0,Q_k),
\qquad
Q_k = \omega_{s_{k-1}}^2(1-a_k^2).
$$

\
This formulation preserves the exact Ornstein-Uhlenbeck transition logic while allowing intraday regime changes. Persistence is inherited from the previous state, but the anchoring level changes with the session.

---

### 15. Why Gaussianity is structurally preserved

The model remains Gaussian because:

1. the stochastic differential equation is linear in the state;
2. the driving Brownian motion has Gaussian increments;
3. the solution is an affine transformation of Gaussian stochastic integrals.

Hence all finite-dimensional distributions are Gaussian whenever the initial law is Gaussian. In particular, the transition density is Gaussian, the stationary law is Gaussian, and the latent-observation system becomes a linear Gaussian state-space model once measurement noise is added.

This is the theoretical reason why Kalman filtering is exact here rather than heuristic.

---

### 16. Summary of the continuous-time theory

The latent spread process is modeled as a mean-reverting diffusion

\
$$
dX_t = \kappa(\mu-X_t)\,dt + \sigma\,dW_t,
$$

\
with exact solution

\
$$
X_t
=
\mu + e^{-\kappa(t-s)}(X_s-\mu)
+
\sigma\int_s^t e^{-\kappa(t-u)}\,dW_u.
$$

\
Its conditional law is

\
$$
X_t \mid X_s
\sim
\mathcal{N}
\left(
\mu + e^{-\kappa(t-s)}(X_s-\mu),
\;
\frac{\sigma^2}{2\kappa}\bigl(1-e^{-2\kappa(t-s)}\bigr)
\right),
$$

\
its invariant law is

\
$$
\mathcal{N}\left(\mu,\frac{\sigma^2}{2\kappa}\right),
$$

\
its autocorrelation function is

\
$$
\rho(h)=e^{-\kappa h},
$$

\
and its half-life is

\
$$
\frac{\ln 2}{\kappa}.
$$

\
Therefore the ***Ornstein-Uhlenbeck process is the natural continuous-time Gaussian model for spreads*** that fluctuate randomly but revert statistically toward a normal liquidity level.


In [ ]:
def kalman_nll_fast(theta: np.ndarray, y: np.ndarray, dt: np.ndarray, sc: np.ndarray) -> float:
    p = unpack_theta(theta)

    if (np.any(~np.isfinite(p.mu)) or (not np.isfinite(p.half_life)) or
        np.any(~np.isfinite(p.stationary_sd)) or np.any(~np.isfinite(p.R))):
        return 1e100
    if p.half_life <= 0 or np.any(p.stationary_sd <= 0) or np.any(p.R <= 0):
        return 1e100

    mu = p.mu
    kappa = p.kappa
    sd = p.stationary_sd
    R = p.R

    y = np.asarray(y, float)
    dt = np.asarray(dt, float)
    sc = np.asarray(sc, int)

    s0 = int(sc[0])
    x = float(mu[s0])
    P = max(float(sd[s0] ** 2), 1e-8)

    nll = 0.0
    two_pi = 2.0 * np.pi

    for k in range(len(y)):
        s_k = int(sc[k])

        if k == 0:
            x_pred, P_pred = x, P
        else:
            s_prev = int(sc[k - 1])
            dt_k = max(float(dt[k]), 1e-9)

            a = np.exp(-kappa * dt_k)
            Q = float(sd[s_prev] ** 2) * (1.0 - a * a)

            # KEY: deviation from previous session mean
            x_pred = float(mu[s_k]) + a * (x - float(mu[s_prev]))
            P_pred = (a * a) * P + Q

        S = P_pred + float(R[s_k])
        if (not np.isfinite(S)) or S <= 0:
            return 1e100

        v = y[k] - x_pred
        nll += 0.5 * (np.log(two_pi * S) + (v * v) / S)

        K = P_pred / S
        x = x_pred + K * v
        P = (1.0 - K) * P_pred
        if P < 1e-12:
            P = 1e-12

    return float(nll) if np.isfinite(nll) else 1e100

## Kalman Filtering, Innovation Likelihood, and ***State-Space model***  Estimation

We now derive the filtering and estimation machinery used for the session-dependent latent spread model. Because the latent state follows an exact Gaussian Ornstein-Uhlenbeck transition and the observed spread is modeled as a noisy Gaussian measurement of that state, the full model is a scalar linear Gaussian state-space system. Therefore the Kalman filter provides the exact recursive conditional distribution of the latent state and the exact Gaussian likelihood of the observations.

---

### 1. State-space representation induced by the Ornstein-Uhlenbeck diffusion

\
Let $ Y_k $ denote the observed spread at observation time $ t_k $, and let $ X_k := X_{t_k} $ denote the latent efficient spread.

\
From the exact Ornstein-Uhlenbeck discretization on an irregular time grid,

\
$$
X_k
=
\mu_{s_k}
+
a_k\bigl(X_{k-1}-\mu_{s_{k-1}}\bigr)
+
\eta_k,
$$

\
where

\
$$
a_k = e^{-\kappa \Delta t_k},
\qquad
\Delta t_k = t_k - t_{k-1},
$$

and

\
$$
\eta_k \sim \mathcal{N}(0,Q_k),
\qquad
Q_k = \omega_{s_{k-1}}^2(1-a_k^2).
$$

\
The observation equation is

\
$$
Y_k = X_k + \varepsilon_k,
\qquad
\varepsilon_k \sim \mathcal{N}(0,R_{s_k}).
$$

\
Assume that:

- $ \eta_k $ are mutually independent,
- $ \varepsilon_k $ are mutually independent,
- $ \eta_k $ are independent of all $ \varepsilon_j $,
- the initial state $ X_1 $ is Gaussian.

Then the pair $ (X_k,Y_k) $ forms a linear Gaussian hidden Markov model.

\
It is convenient to rewrite the state equation in affine linear form:

\
$$
\boxed{
X_k = c_k + a_k X_{k-1} + \eta_k,
}
$$

\
with

\
$$
c_k = \mu_{s_k} - a_k \mu_{s_{k-1}}.
$$

\
The observation equation is

\
$$
\boxed{
Y_k = H X_k + \varepsilon_k,
\qquad H=1.
}
$$

\
Because the model is scalar, all matrices reduce to scalars, but the probabilistic structure is exactly the same as in the general Kalman filter.

---

### 2. Recursive conditional distributions

The main filtering objects are:

- the **prediction law**
  $
  X_k \mid Y_{1:k-1},
  $
- the **filtering law**
  $
  X_k \mid Y_{1:k}.
  $

Since the model is Gaussian, these conditional laws are Gaussian. We write

\
$$
X_k \mid Y_{1:k-1}
\sim \mathcal{N}(m_{k|k-1},P_{k|k-1}),
$$

\
$$
X_k \mid Y_{1:k}
\sim \mathcal{N}(m_{k|k},P_{k|k}).
$$

\
The Kalman filter is precisely the recursion that propagates $ (m_{k|k},P_{k|k}) $ forward in $ k $.

---

### 3. Prediction step from Gaussian conditioning

Assume

\
$$
X_{k-1} \mid Y_{1:k-1}
\sim \mathcal{N}(m_{k-1|k-1},P_{k-1|k-1}).
$$

\
Since

\
$$
X_k = c_k + a_k X_{k-1} + \eta_k,
\qquad
\eta_k \sim \mathcal{N}(0,Q_k),
$$

\
and $ \eta_k $ is independent of $ X_{k-1}\mid Y_{1:k-1} $, it follows that $ X_k \mid Y_{1:k-1} $ is Gaussian with mean

\
$$
\mathbb{E}[X_k\mid Y_{1:k-1}]
=
c_k + a_k\,\mathbb{E}[X_{k-1}\mid Y_{1:k-1}]
=
c_k + a_k m_{k-1|k-1},
$$

\
so

\
$$
\boxed{
m_{k|k-1} = c_k + a_k m_{k-1|k-1}.
}
$$

\
Similarly, the conditional variance is

\
$$
\operatorname{Var}(X_k\mid Y_{1:k-1})
=
a_k^2 \operatorname{Var}(X_{k-1}\mid Y_{1:k-1}) + \operatorname{Var}(\eta_k),
$$

\
hence

\
$$
\boxed{
P_{k|k-1} = a_k^2 P_{k-1|k-1} + Q_k.
}
$$

\
In session notation this is

\
$$
m_{k|k-1}
=
\mu_{s_k}
+
a_k\bigl(m_{k-1|k-1}-\mu_{s_{k-1}}\bigr).
$$

\
This is exactly the recursion implemented in the code.

---

### 4. Forecast distribution of the observation

Because

\
$$
Y_k = X_k + \varepsilon_k,
\qquad
\varepsilon_k \sim \mathcal{N}(0,R_{s_k}),
$$

\
and $ \varepsilon_k $ is independent of $ X_k\mid Y_{1:k-1} $, the forecast law of the observation is

\
$$
Y_k \mid Y_{1:k-1}
\sim
\mathcal{N}(f_k,S_k),
$$

\
with forecast mean

\
$$
\boxed{
f_k = \mathbb{E}[Y_k\mid Y_{1:k-1}] = m_{k|k-1},
}
$$

\
and forecast variance

\
$$
\boxed{
S_k = \operatorname{Var}(Y_k\mid Y_{1:k-1}) = P_{k|k-1} + R_{s_k}.
}
$$

\
Define the innovation

\
$$
\boxed{
v_k = Y_k - f_k = Y_k - m_{k|k-1}.
}
$$

\
Then

\
$$
\boxed{
v_k \mid Y_{1:k-1} \sim \mathcal{N}(0,S_k).
}
$$

\
The innovation is the unpredictable part of the new observation after accounting for all past information.

---

### 5. Joint Gaussian projection derivation of the update step

To derive the posterior law $ X_k \mid Y_{1:k} $, consider the joint Gaussian vector

\
$$
\begin{pmatrix}
X_k \\
Y_k
\end{pmatrix}
\Bigg| Y_{1:k-1}.
$$

\
We already know:

\
$$
X_k \mid Y_{1:k-1} \sim \mathcal{N}(m_{k|k-1},P_{k|k-1}),
$$

\
and

\
$$
Y_k = X_k + \varepsilon_k.
$$

\
Hence, conditional on $ Y_{1:k-1} $,

\
$$
\mathbb{E}[X_k] = m_{k|k-1},
\qquad
\mathbb{E}[Y_k] = m_{k|k-1}.
$$

\
The conditional covariance matrix is

\
$$
\operatorname{Cov}
\left[
\begin{pmatrix}
X_k \\
Y_k
\end{pmatrix}
\Bigg| Y_{1:k-1}
\right]
=
\begin{pmatrix}
P_{k|k-1} & P_{k|k-1} \\
P_{k|k-1} & P_{k|k-1}+R_{s_k}
\end{pmatrix}.
$$

\
For a jointly Gaussian vector, the conditional expectation is linear. Therefore

\
$$
\mathbb{E}[X_k \mid Y_{1:k}]
=
m_{k|k-1}
+
\frac{\operatorname{Cov}(X_k,Y_k\mid Y_{1:k-1})}{\operatorname{Var}(Y_k\mid Y_{1:k-1})}
\bigl(Y_k - m_{k|k-1}\bigr).
$$

\
Substituting the covariance terms gives

\
$$
\mathbb{E}[X_k \mid Y_{1:k}]
=
m_{k|k-1}
+
\frac{P_{k|k-1}}{P_{k|k-1}+R_{s_k}}
\bigl(Y_k - m_{k|k-1}\bigr).
$$

\
Hence

\
$$
\boxed{
m_{k|k}
=
m_{k|k-1}
+
K_k v_k,
}
$$

\
where the Kalman gain is

\
$$
\boxed{
K_k = \frac{P_{k|k-1}}{S_k}.
}
$$

\
The posterior variance of a conditional Gaussian is

\
$$
P_{k|k}
=
P_{k|k-1}
-
\frac{P_{k|k-1}^2}{S_k},
$$

\
so

\
$$
\boxed{
P_{k|k}
=
(1-K_k)P_{k|k-1}.
}
$$

\
This is the exact Bayesian updating formula in the scalar Gaussian case.

---

### 6. Bayes-theorem derivation of the filtering density

The same result may be derived directly from Bayes' formula. Write

\
$$
p(x_k \mid y_{1:k})
\propto
p(y_k \mid x_k)\,p(x_k \mid y_{1:k-1}).
$$

\
Now

\
$$
p(x_k \mid y_{1:k-1})
\propto
\exp\left(
-\frac{1}{2}
\frac{(x_k-m_{k|k-1})^2}{P_{k|k-1}}
\right),
$$

\
and

\
$$
p(y_k \mid x_k)
\propto
\exp\left(
-\frac{1}{2}
\frac{(y_k-x_k)^2}{R_{s_k}}
\right).
$$

\
Therefore

\
$$
p(x_k \mid y_{1:k})
\propto
\exp\left(
-\frac12
\left[
\frac{(x_k-m_{k|k-1})^2}{P_{k|k-1}}
+
\frac{(y_k-x_k)^2}{R_{s_k}}
\right]
\right).
$$

\
Expanding the quadratic form in $ x_k $,

\
$$
\frac{(x_k-m)^2}{P} + \frac{(y-x_k)^2}{R}
=
x_k^2\left(\frac{1}{P}+\frac{1}{R}\right)
-2x_k\left(\frac{m}{P}+\frac{y}{R}\right)
+\text{const},
$$

\
where $ m=m_{k|k-1} $, $ P=P_{k|k-1} $, $ R=R_{s_k} $.

\
Completing the square shows that the posterior is Gaussian with variance

\
$$
\boxed{
P_{k|k}
=
\left(\frac{1}{P_{k|k-1}}+\frac{1}{R_{s_k}}\right)^{-1}
=
\frac{P_{k|k-1}R_{s_k}}{P_{k|k-1}+R_{s_k}},
}
$$

\
and mean

\
$$
\boxed{
m_{k|k}
=
P_{k|k}
\left(
\frac{m_{k|k-1}}{P_{k|k-1}} + \frac{Y_k}{R_{s_k}}
\right).
}
$$

\
After algebraic simplification, this is equivalent to the Kalman update

\
$$
m_{k|k}
=
m_{k|k-1}
+
\frac{P_{k|k-1}}{P_{k|k-1}+R_{s_k}}
\bigl(Y_k-m_{k|k-1}\bigr).
$$

\
Thus the Kalman filter is exactly Bayes' theorem specialized to linear Gaussian dynamics.

---

### 7. Orthogonality principle and minimum mean squared error interpretation

Because the model is Gaussian, the conditional mean is the minimum mean squared error estimator. Thus

\
$$
m_{k|k} = \mathbb{E}[X_k \mid Y_{1:k}]
$$

\
is the unique minimizer of

\
$$
\mathbb{E}\bigl[(X_k-\hat X_k)^2 \mid Y_{1:k}\bigr]
$$

\
over all $ Y_{1:k} $ -measurable estimators $ \hat X_k $.

\
The filtering error

\
$$
e_{k|k} := X_k - m_{k|k}
$$

\
satisfies the orthogonality condition

\
$$
\mathbb{E}\bigl[e_{k|k}\,\phi(Y_{1:k})\bigr]=0
$$

\
for every square-integrable measurable function $ \phi $. In particular,

\
$$
\mathbb{E}\bigl[e_{k|k}Y_j\bigr]=0,
\qquad j\le k.
$$

\
This is another way to understand why the filter is optimal: the residual error is orthogonal to all information already extracted from the observation history.

---

### 8. Innovation process and martingale difference property

Define the innovation process

\
$$
v_k = Y_k - \mathbb{E}[Y_k \mid Y_{1:k-1}].
$$

\
Then:

1. $ v_k $ is $ \sigma(Y_{1:k}) $ -measurable;
2. $ \mathbb{E}[v_k \mid Y_{1:k-1}] = 0 $;
3. $ \operatorname{Var}(v_k \mid Y_{1:k-1}) = S_k $.

Hence $ \{v_k\} $ is a martingale difference sequence with predictable conditional variance $ S_k $. In linear Gaussian models, the innovations are also mutually uncorrelated:

\
$$
\mathbb{E}[v_k v_j] = 0,
\qquad k\neq j.
$$

\
This innovation orthogonality underlies the likelihood factorization.

---

### 9. Innovation decomposition of the likelihood

The joint density of observations admits the chain factorization

\
$$
p(y_1,\dots,y_n)
=
\prod_{k=1}^n p(y_k \mid y_{1:k-1}).
$$

\
Since

\
$$
Y_k \mid Y_{1:k-1} \sim \mathcal{N}(f_k,S_k),
$$

\
each conditional density is

\
$$
p(y_k \mid y_{1:k-1})
=
\frac{1}{\sqrt{2\pi S_k}}
\exp\left(
-\frac{(y_k-f_k)^2}{2S_k}
\right).
$$

\
Because $ v_k = y_k-f_k $, we obtain

\
$$
p(y_1,\dots,y_n)
=
\prod_{k=1}^n
\frac{1}{\sqrt{2\pi S_k}}
\exp\left(
-\frac{v_k^2}{2S_k}
\right).
$$

\
Taking logarithms,

\
$$
\log L(\theta)
=
-\frac12 \sum_{k=1}^n
\left[
\log(2\pi) + \log S_k + \frac{v_k^2}{S_k}
\right].
$$

\
Thus the negative log-likelihood is

\
$$
\boxed{
\mathcal{L}(\theta)
=
\frac12 \sum_{k=1}^n
\left[
\log(2\pi S_k) + \frac{v_k^2}{S_k}
\right].
}
$$

\
This is exactly the objective computed in `kalman_nll_fast(...)`.

---

### 10. Why the Kalman likelihood is exact here

In many time-series settings filtering-based likelihoods are approximations. Here they are exact because:

- the state transition is the exact discrete-time law of the Ornstein-Uhlenbeck diffusion;
- the transition density is Gaussian;
- the observation density is Gaussian;
- the latent model is linear in the state;
- the initial distribution is Gaussian.

Therefore the filtering recursion produces the exact sequence of predictive Gaussian densities $ p(y_k \mid y_{1:k-1}) $, and the resulting log-likelihood is the exact Gaussian likelihood of the observed spread series.

---

### 11. Session-dependent parametrization in the likelihood

The parameter vector is

\
$$
\theta
=
\bigl(
\mu_0,\mu_1,\mu_2,\mu_3,\log h,\log \omega_0,\log \omega_1,\log \omega_2,\log \omega_3,\log R_0,\log R_1,\log R_2,\log R_3
\bigr),
$$


where:

- $ \mu_s $ are session-specific equilibrium spreads,
- $ h $ is the global half-life,
- $ \omega_s $ are session-specific stationary standard deviations of the latent state,
- $ R_s $ are session-specific observation noise variances.

The half-life determines the mean-reversion speed by

\
$$
\kappa = \frac{\ln 2}{h}.
$$

\
Then for each interval $ \Delta t_k $,

\
$$
a_k = e^{-\kappa \Delta t_k},
$$


\
and for each previous session $ s_{k-1} $,

\
$$
Q_k = \omega_{s_{k-1}}^2(1-a_k^2).
$$

\
So the likelihood is highly structured: the same global $ \kappa $ enters every transition coefficient, while session labels select the appropriate $ \mu_s $, $ \omega_s $, and $ R_s $.

---

### 12. Filtering recursion written in the exact notation of the implementation

For clarity, the scalar filter used in the code can be written exactly as follows.

#### Initialization

For the first observation, if $ s_1 $ is the initial session,

\
$$
x_{1|0} = \mu_{s_1},
\qquad
P_{1|0} = \omega_{s_1}^2.
$$

#### Prediction for $ k\ge 2 $

$$
a_k = e^{-\kappa \Delta t_k},
$$

\
$$
Q_k = \omega_{s_{k-1}}^2(1-a_k^2),
$$

\
$$
x_{k|k-1}
=
\mu_{s_k}
+
a_k(x_{k-1|k-1}-\mu_{s_{k-1}}),
$$

\
$$
P_{k|k-1} = a_k^2 P_{k-1|k-1} + Q_k.
$$

#### Innovation

$$
v_k = y_k - x_{k|k-1},
\qquad
S_k = P_{k|k-1} + R_{s_k}.
$$

#### Update

$$
K_k = \frac{P_{k|k-1}}{S_k},
$$

\
$$
x_{k|k} = x_{k|k-1} + K_k v_k,
$$

\
$$
P_{k|k} = (1-K_k)P_{k|k-1}.
$$

#### Negative log-likelihood accumulation

$$
\mathcal{L}
\leftarrow
\mathcal{L}
+
\frac12
\left[
\log(2\pi S_k) + \frac{v_k^2}{S_k}
\right].
$$

This is exactly the operational backbone of the model estimation.

---

### 13. Statistical meaning of the parameters inside the filter

Each parameter influences the filtering recursion in a distinct way.

#### Session mean $ \mu_s $

This shifts the center toward which the latent state is pulled in session $ s $. It affects the deterministic prediction.

#### Half-life $ h $ and mean-reversion speed $ \kappa $

These determine $ a_k = e^{-\kappa \Delta t_k} $, hence persistence. Large $ h $ means slow reversion and more memory. Small $ h $ means rapid forgetting of past deviations.

#### Stationary standard deviation $ \omega_s $

This enters $ Q_k = \omega_{s_{k-1}}^2(1-a_k^2) $, determining how much latent uncertainty accumulates between observations.

#### Observation noise variance $ R_s $

This enters $ S_k = P_{k|k-1} + R_{s_k} $ and therefore controls the Kalman gain

\
$$
K_k = \frac{P_{k|k-1}}{P_{k|k-1}+R_{s_k}}.
$$

If $ R_s $ is large, the filter trusts the observation less. If $ R_s $ is small, the filter follows the observed spread more closely.

---

### 14. Maximum-likelihood estimation

The statistical estimation problem is

\
$$
\boxed{
\hat{\theta}
=
\arg\min_{\theta}
\mathcal{L}(\theta)
=
\arg\min_{\theta}
\frac12 \sum_{k=1}^n
\left[
\log(2\pi S_k(\theta))
+
\frac{v_k(\theta)^2}{S_k(\theta)}
\right].
}
$$

\
This is the exact Gaussian maximum-likelihood estimator of the session-dependent latent Ornstein-Uhlenbeck spread model.

\
Because the filter recursively computes $ v_k(\theta) $ and $ S_k(\theta) $, the likelihood can be evaluated in linear time $ O(n) $, which is computationally essential for high-frequency spread data.

---

### 15. Final theoretical synthesis

The model combines continuous-time stochastic-process theory with exact discrete-time Gaussian inference:

1. the latent efficient spread follows an Ornstein-Uhlenbeck diffusion in continuous time;
2. its exact transition law on an irregular grid is Gaussian;
3. the observed spread equals the latent state plus Gaussian microstructure noise;
4. therefore the model is a linear Gaussian state-space system;
5. the Kalman filter yields the exact recursive conditional distribution of the latent state;
6. the innovation decomposition yields the exact Gaussian log-likelihood;
7. maximum likelihood produces statistically interpretable estimates of equilibrium spreads, reversion speed, latent spread variability, and quote noise.

In this sense, the code implements not a heuristic smoothing device but a fully specified likelihood-based statistical model grounded in diffusion theory, Gaussian state-space analysis, and optimal filtering.

In [ ]:
def fit_sessionwise_ou_fast(
    df: pd.DataFrame,
    spread_col: str = "Spread_pips",
    maxiter: int = 80,
    show_progress: bool = True,
) -> SessionwiseMixedParams:

    y = df[spread_col].values.astype(float)
    dt = df["dt_sec"].values.astype(float)
    sc = df["session_code"].values.astype(int)

    theta0 = make_initial_theta(df, spread_col=spread_col)

    y_min, y_max = float(np.min(y)), float(np.max(y))
    y_range = max(y_max - y_min, 1.0)

    bounds = []
    # mu_s bounds
    for _ in range(4):
        bounds.append((y_min - 0.5 * y_range, y_max + 0.5 * y_range))
    # log half-life bound (global)
    bounds.append((np.log(HALF_LIFE_MIN_SEC), np.log(HALF_LIFE_MAX_SEC)))
    # log sd_s bounds
    for _ in range(4):
        bounds.append((np.log(1e-4), np.log(200.0)))
    # log R_s bounds
    for _ in range(4):
        bounds.append((np.log(1e-8), np.log(1e4)))

    pbar = tqdm(total=maxiter, desc="MLE fast (kappa global)", leave=False) if show_progress else None

    def callback(xk):
        if pbar is not None:
            pp = unpack_theta(xk)
            pbar.set_postfix({
                "HL": f"{pp.half_life:.0f}s",
                "mu_asia": f"{pp.mu[SESSION_TO_INT['Asia']]:.2f}",
                "mu_us": f"{pp.mu[SESSION_TO_INT['US']]:.2f}",
            })
            pbar.update(1)

    res = minimize(
        fun=lambda th: kalman_nll_fast(th, y, dt, sc),
        x0=theta0,
        method="L-BFGS-B",
        bounds=bounds,
        callback=callback,
        options={"maxiter": maxiter, "ftol": 1e-7}
    )

    if pbar is not None:
        pbar.close()

    if not res.success:
        print("WARNING:", res.message)

    return unpack_theta(res.x)

In [ ]:
broker_params: Dict[str, SessionwiseMixedParams] = {}

for broker, df in tqdm(feeds_grid_model.items(), desc="Fitting 13-param model (FAST)"):
    t0 = time.time()
    broker_params[broker] = fit_sessionwise_ou_fast(df, spread_col="Spread_pips", maxiter=80, show_progress=True)
    print(f"{broker}: {time.time() - t0:.1f} sec")

rows = []
for broker, p in broker_params.items():
    rows.append({
        "Broker": broker,
        "half_life_sec": p.half_life,
        "kappa": p.kappa,

        "mu_asia": p.mu[SESSION_TO_INT["Asia"]],
        "mu_europe": p.mu[SESSION_TO_INT["Europe"]],
        "mu_us": p.mu[SESSION_TO_INT["US"]],
        "mu_rollover": p.mu[SESSION_TO_INT["Rollover"]],

        "sd_asia": p.stationary_sd[SESSION_TO_INT["Asia"]],
        "sd_europe": p.stationary_sd[SESSION_TO_INT["Europe"]],
        "sd_us": p.stationary_sd[SESSION_TO_INT["US"]],
        "sd_rollover": p.stationary_sd[SESSION_TO_INT["Rollover"]],

        "R_asia": p.R[SESSION_TO_INT["Asia"]],
        "R_europe": p.R[SESSION_TO_INT["Europe"]],
        "R_us": p.R[SESSION_TO_INT["US"]],
        "R_rollover": p.R[SESSION_TO_INT["Rollover"]],
    })

broker_param_table = pd.DataFrame(rows).sort_values("Broker").reset_index(drop=True)
display(broker_param_table)

## Interpretation of the Estimated Parameters and Comparative Conclusions Across Liquidity Providers

After calibrating the session-dependent ***Ornstein-Uhlenbeck state-space model*** for each liquidity provider, we obtain estimates of the following quantities:

- the half-life of mean reversion:
  $
  h,
  $
- the mean-reversion speed:
  $
  \kappa = \frac{\ln 2}{h},
  $
- the session-specific long-run spread levels:
  $
  \mu_{\mathrm{Asia}},\quad
  \mu_{\mathrm{Europe}},\quad
  \mu_{\mathrm{US}},\quad
  \mu_{\mathrm{Rollover}},
  $
- the session-specific stationary standard deviations of the latent spread:
  $
  \omega_{\mathrm{Asia}},\quad
  \omega_{\mathrm{Europe}},\quad
  \omega_{\mathrm{US}},\quad
  \omega_{\mathrm{Rollover}},
  $
- the session-specific observation-noise variances:
  $
  R_{\mathrm{Asia}},\quad
  R_{\mathrm{Europe}},\quad
  R_{\mathrm{US}},\quad
  R_{\mathrm{Rollover}}.
  $

\
These estimates allow us to compare the providers not only in terms of average spread tightness, but also in terms of persistence of spread dislocations, latent spread instability, and the noisiness of observed quotes.

---

### Mean-reversion speed and persistence of spread dislocations

The estimated half-lives are:

\
$
h^{(1)} = 677.268 \text{ sec}, \qquad
h^{(2)} = 1370.021 \text{ sec}, \qquad
h^{(3)} = 315.658 \text{ sec}.
$

\
Equivalently, the corresponding mean-reversion speeds are

\
$
\kappa^{(1)} = 0.001023, \qquad
\kappa^{(2)} = 0.000506, \qquad
\kappa^{(3)} = 0.002196.
$

\
These values imply a clear ranking in terms of ***how quickly the spread returns toward its session-specific equilibrium*** after a temporary shock:

\
$
\kappa^{(3)} > \kappa^{(1)} > \kappa^{(2)}
\qquad\Longleftrightarrow\qquad
h^{(3)} < h^{(1)} < h^{(2)}.
$

\
Therefore:

- **Liquidity Provider \#3** exhibits ***the fastest mean reversion*** and thus the shortest persistence of spread shocks;
- **Liquidity Provider \#1** occupies an intermediate position;
- **Liquidity Provider \#2** exhibits the slowest mean reversion and therefore the most persistent spread dislocations.

From the perspective of client safety, a larger half-life is undesirable because widened-spread states remain active for longer. Hence, with respect to persistence alone, **Liquidity Provider \#2 appears to be the weakest**, while **Liquidity Provider \#3 appears the strongest**.

---

### Session-specific equilibrium spread levels

The estimated long-run spread levels for **Liquidity Provider \#1** are:

\
$
\mu_{\mathrm{Asia}}^{(1)} = 2.338,
\qquad
\mu_{\mathrm{Europe}}^{(1)} = 1.513,
\qquad
\mu_{\mathrm{US}}^{(1)} = 1.924,
\qquad
\mu_{\mathrm{Rollover}}^{(1)} = 1.171.
$

\
The estimated long-run spread levels for **Liquidity Provider \#2** are:

\
$
\mu_{\mathrm{Asia}}^{(2)} = 32.383,
\qquad
\mu_{\mathrm{Europe}}^{(2)} = 21.934,
\qquad
\mu_{\mathrm{US}}^{(2)} = 25.672,
\qquad
\mu_{\mathrm{Rollover}}^{(2)} = 22.000.
$

\
The estimated long-run spread levels for **Liquidity Provider \#3** are:

\
$
\mu_{\mathrm{Asia}}^{(3)} = 81.976,
\qquad
\mu_{\mathrm{Europe}}^{(3)} = 60.170,
\qquad
\mu_{\mathrm{US}}^{(3)} = 62.716,
\qquad
\mu_{\mathrm{Rollover}}^{(3)} = 70.000.
$

\
These estimates immediately show that the providers are separated by very large differences in their equilibrium spread regimes. In every session,

\
$
\mu_{s}^{(1)} < \mu_{s}^{(2)} < \mu_{s}^{(3)},
\qquad
s \in \{\mathrm{Asia},\mathrm{Europe},\mathrm{US},\mathrm{Rollover}\}.
$

\
Thus:

- **Liquidity Provider \#1** delivers by far the tightest spread environment in all sessions;
- **Liquidity Provider \#2** is materially wider than Provider \#1;
- **Liquidity Provider \#3** is the widest provider in every session by a large margin.

From the standpoint of direct transaction costs and execution quality, this is the most important result: **Provider \#1 dominates the others in terms of equilibrium spread tightness**.

---

### Intraday structure of equilibrium spreads

For **Liquidity Provider \#1**, the session pattern is

\
$
\mu_{\mathrm{Rollover}}^{(1)}
<
\mu_{\mathrm{Europe}}^{(1)}
<
\mu_{\mathrm{US}}^{(1)}
<
\mu_{\mathrm{Asia}}^{(1)}.
$

\
Hence its widest equilibrium spread appears during the Asian session, while the tightest level is observed during rollover.

\
For **Liquidity Provider \#2**, the session ordering is approximately

\
$
\mu_{\mathrm{Europe}}^{(2)}
<
\mu_{\mathrm{Rollover}}^{(2)}
\approx
\mu_{\mathrm{US}}^{(2)}
<
\mu_{\mathrm{Asia}}^{(2)}.
$

\
Thus Provider \#2 is also weakest in Asia and relatively tighter in Europe.

\
For **Liquidity Provider \#3**, the ordering is

\
$
\mu_{\mathrm{Europe}}^{(3)}
<
\mu_{\mathrm{US}}^{(3)}
<
\mu_{\mathrm{Rollover}}^{(3)}
<
\mu_{\mathrm{Asia}}^{(3)}.
$

\
Again, the Asian session is the most adverse regime, but in this case the rollover session is also extremely wide.

\
A common structural feature across all providers is therefore:

\
$
\mu_{\mathrm{Asia}} \text{ is among the highest equilibrium spread levels.}
$

\
This suggests that ***the Asian session is consistently the most fragile regime in terms of quoted spread quality***.

---

### Stationary dispersion of the latent spread

The estimated stationary standard deviations of the latent spread for **Liquidity Provider \#1** are

\
$
\omega_{\mathrm{Asia}}^{(1)} = 3.330,
\qquad
\omega_{\mathrm{Europe}}^{(1)} = 1.184,
\qquad
\omega_{\mathrm{US}}^{(1)} = 2.317,
\qquad
\omega_{\mathrm{Rollover}}^{(1)} = 0.246.
$

\
For **Liquidity Provider \#2**:

\
$
\omega_{\mathrm{Asia}}^{(2)} = 76.661,
\qquad
\omega_{\mathrm{Europe}}^{(2)} = 3.063,
\qquad
\omega_{\mathrm{US}}^{(2)} = 50.403,
\qquad
\omega_{\mathrm{Rollover}}^{(2)} = 7.576.
$

\
For **Liquidity Provider \#3**:

\
$
\omega_{\mathrm{Asia}}^{(3)} = 14.186,
\qquad
\omega_{\mathrm{Europe}}^{(3)} = 11.887,
\qquad
\omega_{\mathrm{US}}^{(3)} = 28.919,
\qquad
\omega_{\mathrm{Rollover}}^{(3)} = 10.531.
$

\
These quantities measure the ***intrinsic variability of the latent spread process around its session-specific equilibrium***. Large values of $ \omega_s $ imply that even ***after removing observation noise, the underlying spread state itself remains unstable***.

\
The most striking result is the extremely large latent dispersion of **Liquidity Provider \#2** during Asia and the US:

\
$
\omega_{\mathrm{Asia}}^{(2)} = 76.661,
\qquad
\omega_{\mathrm{US}}^{(2)} = 50.403.
$

\
***This indicates that Provider \#2 is not only wide on average, but also structurally unstable during these sessions***.

\
By contrast, **Liquidity Provider \#1** exhibits the lowest latent variability in all sessions, especially in Europe and rollover. Therefore **Provider \#1 is the most stable in latent spread terms**.

\
For **Liquidity Provider \#3**, the latent variability is high but not uniformly catastrophic. Its main problem is not persistence, since $ h^{(3)} $ is relatively low, but rather the combination of very high equilibrium spreads and substantial sessional volatility.

---

### Observation-noise variance and quote noisiness

The estimated observation-noise variances for **Liquidity Provider \#1** are

\
$
R_{\mathrm{Asia}}^{(1)} = 29.627,
\qquad
R_{\mathrm{Europe}}^{(1)} = 12.621,
\qquad
R_{\mathrm{US}}^{(1)} = 25.687,
\qquad
R_{\mathrm{Rollover}}^{(1)} = 4.687.
$

\
For **Liquidity Provider \#2**:

\
$
R_{\mathrm{Asia}}^{(2)} = 31.115,
\qquad
R_{\mathrm{Europe}}^{(2)} = 2.424,
\qquad
R_{\mathrm{US}}^{(2)} = 22.330,
\qquad
R_{\mathrm{Rollover}}^{(2)} = 28.700.
$

\
For **Liquidity Provider \#3**:

\
$
R_{\mathrm{Asia}}^{(3)} = 52.225,
\qquad
R_{\mathrm{Europe}}^{(3)} = 66.291,
\qquad
R_{\mathrm{US}}^{(3)} = 43.340,
\qquad
R_{\mathrm{Rollover}}^{(3)} = 55.449.
$

\
The parameter $ R_s $ measures ***the variance of short-lived quote noise around the latent spread level***. Large values imply that the observed spread is noisy, flickering, or contaminated by short-horizon microstructure disturbances.

\
In this dimension, **Liquidity Provider \#3 is clearly the noisiest provider** across all sessions. In particular,

\
$
R_{\mathrm{Europe}}^{(3)} = 66.291
$

\
is the largest observation-noise estimate in the entire table. This suggests that even beyond its very high equilibrium spread, Provider \#3 also produces highly irregular observed spread behavior.

\
**Liquidity Provider \#1** has moderate observation noise, while **Liquidity Provider \#2** looks mixed: Europe appears relatively clean in terms of $ R_{\mathrm{Europe}}^{(2)} $, but Asia and rollover remain materially noisy.

---

### Provider-level summary

Putting all dimensions together:

#### Liquidity Provider \#1

This provider has:

\
$
\min_s \mu_s \text{ across providers}, \qquad
\min_s \omega_s \text{ across providers in most sessions},
$

\
together with an intermediate half-life

\
$
h^{(1)} = 677.268 \text{ sec}.
$

\
Hence Provider \#1 combines ***tight equilibrium spreads with relatively low latent instability and moderate quote noise. From the client-safety perspective, this is the strongest overall profile in the sample***.

#### Liquidity Provider \#2

This provider has:

- much wider equilibrium spreads than Provider \#1,
- the largest half-life
  $
  h^{(2)} = 1370.021 \text{ sec},
  $
- extremely large latent dispersion in Asia and the US.

Thus Provider \#2 appears ***especially problematic because spread shocks persist for a long time and the latent spread process itself is highly unstable during key sessions***. Even where the observed quote noise is not the worst, ***the persistence component makes this provider dangerous from a risk-management perspective***.

#### Liquidity Provider \#3

This provider has:

- the largest equilibrium spreads in every session,
- the fastest mean reversion
  $
  h^{(3)} = 315.658 \text{ sec},
  $
- the largest observation-noise variances.

Hence Provider \#3 is ***structurally very wide and very noisy, but relatively fast in reverting toward its own equilibrium***. This means that its problem is less about persistence and more about the fact that its normal operating regime is already unfavorable for the client.

---

### Final ranking from the perspective of client trading safety

If the objective is to identify the provider offering the safest spread environment for clients, the estimates suggest the following qualitative ranking:

\
$
\boxed{
\text{Liquidity Provider #1} \succ \text{Liquidity Provider #2} \succ \text{Liquidity Provider #3}
}
$

\
when prioritizing **equilibrium spread tightness and latent stability**, and

\
$
\boxed{
\text{Liquidity Provider #1} \succ \text{Liquidity Provider #3} \succ \text{Liquidity Provider #2}
}
$

\
when prioritizing **speed of recovery after spread shocks**.

\
Therefore, the final assessment depends on which dimension is considered most critical. However, from a broad client-protection perspective, the dominant conclusion is:

\
$
\boxed{
\text{Liquidity Provider #1 is the strongest overall candidate.}
}
$

\
It combines the tightest equilibrium spreads with the lowest latent instability and acceptable persistence.

\
At the opposite end, the weakest provider depends on the criterion:

- **Liquidity Provider \#2** is the weakest in terms of persistence and structural instability of spread shocks;
- **Liquidity Provider \#3** is the weakest in terms of absolute spread level and quote noisiness.

Thus the model reveals that poor spread quality may arise through different mechanisms: either through **long-lived spread deterioration** or through **structurally wide and noisy quoting**.

## Visualization of Spread Expansions Across Trading Days

Having obtained a full parametric characterization of ***the spread dynamics*** for each liquidity provider, we now turn to a more direct empirical analysis of spread behavior in calendar time. In particular, we focus on identifying and visualizing spread expansion events across different days of the week.

\
While the ***Ornstein-Uhlenbeck state-space model*** provides a compact statistical description of the latent spread process through parameters such as $ \mu_s $, $ \kappa $, $ \omega_s $, and $ R_s $, it is also important to examine how these dynamics manifest in observable time series.

To this end, we consider the observed spread process

\
$$
Y_t = X_t + \varepsilon_t,
$$

\
and construct empirical diagnostics of spread expansions based on the following characteristics:

- **amplitude of spread expansion**, defined as deviations of $ Y_t $ from its local baseline level;
- **duration of expansion episodes**, measured through consecutive time intervals where the spread remains elevated;
- **frequency of such events**, capturing how often abnormal spread regimes occur;
- **distribution across trading sessions and weekdays**.

These metrics complement the parametric model by providing a direct view of how often and how severely the spread deviates from its equilibrium level.

\
***From a stochastic-process perspective***, spread expansions correspond to realizations where the latent state $ X_t $ deviates significantly from its conditional mean

\
$$
\mathbb{E}[X_t \mid \mathcal{F}_{t^-}] = \mu_{s_t} + e^{-\kappa \Delta t}(X_{t^-} - \mu_{s_{t^-}}),
$$

\
and remains in that region for a non-trivial time interval.

\
Thus, the visualization step serves as an empirical validation layer, linking the theoretical structure of the ***Ornstein-Uhlenbeck model*** with observable ***spread-risk patterns across time***.

In [ ]:
import numpy as np
import pandas as pd

def filter_and_smooth_fast(
    df: pd.DataFrame,
    params,  # SessionwiseMixedParams
    spread_col: str = "Spread_pips"
) -> pd.DataFrame:
    """
    Kalman filter + RTS smoother для модели:
      x_k = mu[s_k] + a_k (x_{k-1} - mu[s_{k-1}]) + eta_k
      a_k = exp(-kappa * dt_k)
      Var(eta_k) = sd[s_{k-1}]^2 * (1 - a_k^2)
      y_k = x_k + eps_k, Var(eps_k)=R[s_k]

    Возвращает df с:
      session_mean_pips, state_pred_pips, state_filt_pips, state_smooth_pips,
      P_pred/P_filt/P_smooth, innovation/innovation_var/innovation_z,
      state_gap_pips.
    """
    xdf = df.copy().sort_values("Date").reset_index(drop=True)

    y = xdf[spread_col].values.astype(float)
    dt = xdf["dt_sec"].values.astype(float)
    sc = xdf["session_code"].values.astype(int)
    n = len(xdf)

    mu = params.mu
    kappa = params.kappa
    sd = params.stationary_sd
    R = params.R

    x_pred = np.full(n, np.nan)
    P_pred = np.full(n, np.nan)
    x_filt = np.full(n, np.nan)
    P_filt = np.full(n, np.nan)
    a_arr  = np.full(n, np.nan)

    innov = np.full(n, np.nan)
    innov_var = np.full(n, np.nan)

    # init (стационарно в первой сессии)
    s0 = int(sc[0])
    x = float(mu[s0])
    P = max(float(sd[s0] ** 2), 1e-8)

    for k in range(n):
        s_k = int(sc[k])

        if k == 0:
            xp, Pp = x, P
            a_arr[k] = np.nan
        else:
            s_prev = int(sc[k - 1])
            dt_k = max(float(dt[k]), 1e-9)

            a = np.exp(-float(kappa) * dt_k)
            Q = float(sd[s_prev] ** 2) * (1.0 - a * a)

            xp = float(mu[s_k]) + a * (x - float(mu[s_prev]))
            Pp = (a * a) * P + Q
            a_arr[k] = a

        S = Pp + float(R[s_k])
        v = y[k] - xp
        K = Pp / S
        xf = xp + K * v
        Pf = (1.0 - K) * Pp

        x_pred[k], P_pred[k] = xp, Pp
        x_filt[k], P_filt[k] = xf, Pf
        innov[k], innov_var[k] = v, S

        x = float(xf)
        P = max(float(Pf), 1e-12)

    # RTS smoother
    x_smooth = np.full(n, np.nan)
    P_smooth = np.full(n, np.nan)
    x_smooth[-1] = x_filt[-1]
    P_smooth[-1] = P_filt[-1]

    for k in range(n - 2, -1, -1):
        a_next = float(a_arr[k + 1])
        denom = max(float(P_pred[k + 1]), 1e-12)
        Ck = float(P_filt[k]) * a_next / denom

        x_smooth[k] = x_filt[k] + Ck * (x_smooth[k + 1] - x_pred[k + 1])
        P_smooth[k] = P_filt[k] + (Ck**2) * (P_smooth[k + 1] - P_pred[k + 1])

    out = xdf.copy()
    out["session_mean_pips"] = mu[sc]
    out["state_pred_pips"] = x_pred
    out["state_filt_pips"] = x_filt
    out["state_smooth_pips"] = x_smooth

    out["P_pred"] = P_pred
    out["P_filt"] = P_filt
    out["P_smooth"] = P_smooth

    out["innovation"] = innov
    out["innovation_var"] = innov_var
    out["innovation_z"] = out["innovation"] / np.sqrt(out["innovation_var"])

    out["state_gap_pips"] = out["state_smooth_pips"] - out["session_mean_pips"]

    # сессионный stationary sd (для порогов)
    out["stationary_sd_pips"] = sd[sc]
    out["half_life_sec"] = params.half_life  # константа на весь ряд

    return out

In [ ]:
from tqdm.auto import tqdm

processed_model_parts = []
processed_viz_parts = []

for broker in tqdm(broker_params.keys(), desc="Filtering + smoothing (13-param)"):
    p = broker_params[broker]

    out_model = filter_and_smooth_fast(feeds_grid_model[broker], p, "Spread_pips")
    out_model["Broker"] = broker
    processed_model_parts.append(out_model)

    out_viz = filter_and_smooth_fast(feeds_grid_viz[broker], p, "Spread_pips")
    out_viz["Broker"] = broker
    processed_viz_parts.append(out_viz)

processed_model_all = (
    pd.concat(processed_model_parts, ignore_index=True)
      .sort_values(["Date", "Broker"])
      .reset_index(drop=True)
)

processed_viz_all = (
    pd.concat(processed_viz_parts, ignore_index=True)
      .sort_values(["Date", "Broker"])
      .reset_index(drop=True)
)

print("processed_model_all:", processed_model_all.shape)
print("processed_viz_all  :", processed_viz_all.shape)

In [ ]:
def seconds_to_steps(df: pd.DataFrame, window_sec: int) -> int:
    dt = df["dt_sec"].values
    dt = dt[(dt > 0) & np.isfinite(dt)]
    step = float(np.median(dt)) if len(dt) else 1.0
    return int(max(1, round(window_sec / max(step, 1e-9))))

def add_model_regime_features(
    df: pd.DataFrame,
    persistence_window_sec: int = 180,
    integral_window_sec: int = 180,
    gap_scale_mult: float = 0.75
) -> pd.DataFrame:
    x = df.copy().sort_values("Date").reset_index(drop=True)
    if len(x) == 0:
        return x

    thr = gap_scale_mult * x["stationary_sd_pips"].astype(float)
    x["gap_threshold_pips"] = thr
    x["wide_gap_indicator"] = (x["state_gap_pips"] > thr).astype(int)

    w_pers = seconds_to_steps(x, persistence_window_sec)
    x["wide_persistence"] = x["wide_gap_indicator"].rolling(
        w_pers, min_periods=max(10, w_pers // 5)
    ).mean()

    x["positive_gap_pips"] = np.maximum(x["state_gap_pips"].astype(float), 0.0)
    x["gap_integrand"] = x["positive_gap_pips"] * np.maximum(x["dt_sec"].astype(float), 1e-9)

    w_int = seconds_to_steps(x, integral_window_sec)
    x["rolling_integrated_gap_pipsec"] = x["gap_integrand"].rolling(
        w_int, min_periods=max(10, w_int // 5)
    ).sum()

    return x

# применяем на viz (можно и на model, но для визуализации нужен viz)
processed_viz_all = (
    processed_viz_all.groupby("Broker", sort=True, group_keys=False)
    .apply(lambda g: add_model_regime_features(g, 180, 180, 0.75))
    .sort_values(["Date", "Broker"])
    .reset_index(drop=True)
)

In [ ]:
def detect_expansion_episodes_block(
    df_block: pd.DataFrame,
    gap_enter_mult: float = 0.60,
    gap_exit_mult: float = 0.25,
    persistence_enter: float = 0.55,
    persistence_exit: float = 0.20,
    min_duration_sec: float = 60.0
) -> pd.DataFrame:
    x = df_block.copy().sort_values("Date").reset_index(drop=True)
    if len(x) == 0:
        return pd.DataFrame()

    in_ep = False
    start_idx = None
    peak_idx = None
    peak_gap = -np.inf
    events = []

    for i in range(len(x)):
        pers = x.loc[i, "wide_persistence"]
        gap = x.loc[i, "state_gap_pips"]
        sd_i = x.loc[i, "stationary_sd_pips"]

        if pd.isna(pers) or pd.isna(gap) or pd.isna(sd_i):
            continue

        enter_thr = gap_enter_mult * float(sd_i)
        exit_thr  = gap_exit_mult  * float(sd_i)

        if not in_ep:
            if pers >= persistence_enter and gap >= enter_thr:
                in_ep = True
                start_idx = i
                peak_idx = i
                peak_gap = float(gap)
        else:
            if float(gap) > peak_gap:
                peak_gap = float(gap)
                peak_idx = i

            if pers <= persistence_exit and gap <= exit_thr:
                end_idx = i

                start_time = x.loc[start_idx, "Date"]
                peak_time = x.loc[peak_idx, "Date"]
                end_time = x.loc[end_idx, "Date"]
                duration_sec = float((end_time - start_time).total_seconds())

                if duration_sec >= min_duration_sec:
                    # integrated gap (pip·sec)
                    sl = x.loc[start_idx:end_idx].copy()
                    dts = sl["Date"].diff().dt.total_seconds().fillna(0.0).values
                    positive_gap = np.maximum(sl["state_gap_pips"].values.astype(float), 0.0)
                    integrated_gap = float(np.sum(positive_gap * np.maximum(dts, 1e-9)))

                    events.append({
                        "start_time": start_time,
                        "peak_time": peak_time,
                        "end_time": end_time,
                        "duration_sec": duration_sec,
                        "recovery_time_sec": float((end_time - peak_time).total_seconds()),
                        "peak_gap_pips": float(peak_gap),
                        "integrated_gap_pipsec": integrated_gap,
                        "peak_spread_pips": float(x.loc[peak_idx, "Spread_pips"]),
                    })

                in_ep = False
                start_idx = None
                peak_idx = None
                peak_gap = -np.inf

    # IMPORTANT: если эпизод начался, но не закрылся до конца ряда — закрываем на последней точке
    if in_ep and start_idx is not None:
        end_idx = len(x) - 1
        start_time = x.loc[start_idx, "Date"]
        end_time = x.loc[end_idx, "Date"]
        duration_sec = float((end_time - start_time).total_seconds())
        if duration_sec >= min_duration_sec:
            peak_time = x.loc[peak_idx, "Date"]
            sl = x.loc[start_idx:end_idx].copy()
            dts = sl["Date"].diff().dt.total_seconds().fillna(0.0).values
            positive_gap = np.maximum(sl["state_gap_pips"].values.astype(float), 0.0)
            integrated_gap = float(np.sum(positive_gap * np.maximum(dts, 1e-9)))
            events.append({
                "start_time": start_time,
                "peak_time": peak_time,
                "end_time": end_time,
                "duration_sec": duration_sec,
                "recovery_time_sec": float((end_time - peak_time).total_seconds()),
                "peak_gap_pips": float(peak_gap),
                "integrated_gap_pipsec": integrated_gap,
                "peak_spread_pips": float(x.loc[peak_idx, "Spread_pips"]),
            })

    return pd.DataFrame(events)

def add_episode_severity(ep: pd.DataFrame) -> pd.DataFrame:
    if ep is None or len(ep) == 0:
        return pd.DataFrame()

    out = ep.copy()

    def robust_scale(s: pd.Series) -> pd.Series:
        s = s.astype(float)
        med = s.median()
        mad = (s - med).abs().median()
        scale = 1.4826 * mad if mad > 1e-12 else max(s.std(), 1e-12)
        return (s - med) / scale

    z_peak = robust_scale(out["peak_gap_pips"].fillna(0))
    z_int  = robust_scale(out["integrated_gap_pipsec"].fillna(0))
    z_rec  = robust_scale(out["recovery_time_sec"].fillna(0))

    out["severity_score_raw"] = 0.45*z_peak + 0.40*z_int + 0.15*z_rec
    s = out["severity_score_raw"]
    s_min, s_max = s.min(), s.max()
    out["severity_score"] = (s - s_min) / (s_max - s_min) if (np.isfinite(s_min) and np.isfinite(s_max) and s_max > s_min) else 0.5
    out["is_heavy"] = out["severity_score"] >= out["severity_score"].quantile(0.80)
    return out

episode_parts = []
for broker, g in tqdm(processed_viz_all.groupby("Broker", sort=True), desc="Detecting episodes on 1S viz"):
    ep = detect_expansion_episodes_block(
        g,
        gap_enter_mult=0.60,
        gap_exit_mult=0.25,
        persistence_enter=0.55,
        persistence_exit=0.20,
        min_duration_sec=60.0
    )
    if len(ep) > 0:
        ep["Broker"] = broker
        episode_parts.append(ep)

episodes_viz_all = pd.concat(episode_parts, ignore_index=True) if len(episode_parts) else pd.DataFrame()
episodes_viz_all = add_episode_severity(episodes_viz_all)

print("episodes_viz_all:", episodes_viz_all.shape)
display(episodes_viz_all.head())

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def downsample_for_plot(df: pd.DataFrame, max_points: int = 1400) -> pd.DataFrame:
    if len(df) <= max_points:
        return df
    idx = np.linspace(0, len(df) - 1, max_points).astype(int)
    return df.iloc[idx].copy()

def axis_domain_ref(row_num: int) -> str:
    return "y domain" if row_num == 1 else f"y{row_num} domain"

def build_episode_segment_shapes_clipped(
    block_df: pd.DataFrame,
    plot_start: pd.Timestamp,
    plot_end: pd.Timestamp,
    peak_gap_pips: float,
    is_heavy: bool,
    row_num: int,
    n_segments: int = 8,
    min_alpha: float = 0.05,
    max_alpha: float = 0.30,
    heavy_extra_alpha: float = 0.08
):
    shapes = []
    if block_df is None or len(block_df) < 2:
        return shapes

    ps = pd.to_datetime(plot_start)
    pe = pd.to_datetime(plot_end)
    if pe <= ps:
        return shapes

    seg_df = block_df[(block_df["Date"] >= ps) & (block_df["Date"] <= pe)].copy()
    if len(seg_df) < 2:
        return shapes

    seg_df = seg_df.sort_values("Date").reset_index(drop=True)
    seg_df["gap_pos"] = np.maximum(seg_df["state_gap_pips"].astype(float), 0.0)

    peak_gap = max(float(peak_gap_pips), 1e-8)
    n_segments = max(1, min(int(n_segments), len(seg_df) - 1))
    idx_edges = np.linspace(0, len(seg_df) - 1, n_segments + 1).astype(int)

    for j in range(n_segments):
        part = seg_df.iloc[idx_edges[j]:idx_edges[j+1] + 1]
        if len(part) == 0:
            continue

        x0 = part["Date"].iloc[0]
        x1 = part["Date"].iloc[-1]

        local_gap = float(part["gap_pos"].mean())
        strength = np.clip(local_gap / peak_gap, 0.0, 1.0)

        alpha = min_alpha + (max_alpha - min_alpha) * strength
        if is_heavy:
            alpha = min(alpha + heavy_extra_alpha, 0.50)

        fillcolor = f"rgba(255, 77, 109, {alpha:.3f})" if is_heavy else f"rgba(255, 159, 67, {alpha:.3f})"

        shapes.append(dict(
            type="rect",
            xref="x",
            yref=axis_domain_ref(row_num),
            x0=x0, x1=x1,
            y0=0, y1=1,
            fillcolor=fillcolor,
            line=dict(color="rgba(0,0,0,0)", width=0.0),
            layer="below"
        ))
    return shapes

BROKER_STYLE = {
    "Liquidity Provider #1": {"bid": "#7aa2ff", "ask": "#ff7aa2", "band_fill": "rgba(160,160,255,0.12)", "state": "#b072ff"},
    "Liquidity Provider #2": {"bid": "#72f1b8", "ask": "#ff7aa2", "band_fill": "rgba(120,255,200,0.10)", "state": "#b072ff"},
    "Liquidity Provider #3": {"bid": "#f2e94e", "ask": "#ff7aa2", "band_fill": "rgba(255,240,120,0.10)", "state": "#b072ff"},
    "default": {"bid": "#7aa2ff", "ask": "#ff7aa2", "band_fill": "rgba(160,160,255,0.12)", "state": "#b072ff"},
}

In [ ]:
def build_three_lp_gap_dashboard_clean_fixed(
    processed_all: pd.DataFrame,
    episodes_all: pd.DataFrame,
    max_points_per_broker_day: int = 1400,
    max_events_per_broker_day: int = 200,
    n_episode_segments: int = 8
):
    data = processed_all.copy().sort_values(["trade_date", "Broker", "Date"]).reset_index(drop=True)
    if len(data) == 0:
        raise ValueError("processed_all is empty")

    need_cols = ["Date", "Broker", "trade_date", "Spread_pips", "state_gap_pips"]
    for c in need_cols:
        if c not in data.columns:
            raise ValueError(f"processed_all must contain '{c}'")

    # centered band
    data["Bid_band_pips"] = -data["Spread_pips"] / 2.0
    data["Ask_band_pips"] =  data["Spread_pips"] / 2.0

    brokers = [b for b in ["Liquidity Provider #1", "Liquidity Provider #2", "Liquidity Provider #3"]
               if b in data["Broker"].dropna().unique()]
    dates = sorted(data["trade_date"].dropna().astype(str).unique().tolist())
    if not dates:
        raise ValueError("No dates found")

    subplot_titles = []
    for broker in brokers:
        subplot_titles.append(f"{broker} — Centered spread band")
        subplot_titles.append(f"{broker} — Model widening metric: state_gap_pips")

    row_heights = []
    for _ in brokers:
        row_heights.extend([0.18, 0.14])

    fig = make_subplots(
        rows=2 * len(brokers),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.025,
        row_heights=row_heights,
        subplot_titles=tuple(subplot_titles)
    )

    trace_map = []
    layout_shapes_map = {}
    trace_idx = 0

    ep = episodes_all.copy() if episodes_all is not None else pd.DataFrame()
    if len(ep):
        for col in ["start_time", "end_time", "peak_time"]:
            if col in ep.columns:
                ep[col] = pd.to_datetime(ep[col], errors="coerce")

    for d in dates:
        day_start = pd.to_datetime(d)
        day_end = day_start + pd.Timedelta(days=1)
        shapes_for_day = []

        for i, broker in enumerate(brokers):
            row_band = 2 * i + 1
            row_gap  = 2 * i + 2

            block = data[(data["trade_date"].astype(str) == d) & (data["Broker"] == broker)].copy().sort_values("Date")
            block_plot = downsample_for_plot(block, max_points=max_points_per_broker_day) if len(block) else block

            # ===== FIX: episodes by overlap with day =====
            if len(ep):
                ep_day = ep[
                    (ep["Broker"] == broker) &
                    (ep["start_time"] < day_end) &
                    (ep["end_time"] > day_start)
                ].copy().sort_values("start_time")
            else:
                ep_day = pd.DataFrame()

            if len(ep_day) > max_events_per_broker_day:
                if "severity_score" in ep_day.columns:
                    ep_day = ep_day.sort_values("severity_score", ascending=False).head(max_events_per_broker_day)
                    ep_day = ep_day.sort_values("start_time")
                else:
                    ep_day = ep_day.head(max_events_per_broker_day)

            heavy = ep_day[ep_day.get("is_heavy", False) == True].copy() if len(ep_day) else pd.DataFrame()
            heavy_in_day = heavy[(heavy["peak_time"] >= day_start) & (heavy["peak_time"] < day_end)].copy() if len(heavy) else pd.DataFrame()

            style = BROKER_STYLE.get(broker, BROKER_STYLE["default"])

            # --- band low
            fig.add_trace(
                go.Scatter(
                    x=block_plot["Date"] if len(block_plot) else [],
                    y=block_plot["Bid_band_pips"] if len(block_plot) else [],
                    mode="lines",
                    line=dict(color=style["bid"], width=0.8),
                    showlegend=False,
                    visible=(d == dates[0]),
                    hovertemplate="Time=%{x}<br>Band low=%{y:.2f} pips<extra></extra>"
                ),
                row=row_band, col=1
            )
            trace_map.append((d, broker, "band_low", trace_idx)); trace_idx += 1

            # --- band high + fill
            fig.add_trace(
                go.Scatter(
                    x=block_plot["Date"] if len(block_plot) else [],
                    y=block_plot["Ask_band_pips"] if len(block_plot) else [],
                    mode="lines",
                    line=dict(color=style["ask"], width=0.8),
                    fill="tonexty",
                    fillcolor=style["band_fill"],
                    showlegend=False,
                    visible=(d == dates[0]),
                    hovertemplate="Time=%{x}<br>Band high=%{y:.2f} pips<extra></extra>"
                ),
                row=row_band, col=1
            )
            trace_map.append((d, broker, "band_high", trace_idx)); trace_idx += 1

            # --- gap
            fig.add_trace(
                go.Scattergl(
                    x=block_plot["Date"] if len(block_plot) else [],
                    y=block_plot["state_gap_pips"] if len(block_plot) else [],
                    mode="lines",
                    line=dict(color=style["state"], width=1.7),
                    showlegend=False,
                    visible=(d == dates[0]),
                    hovertemplate="Time=%{x}<br>gap=%{y:.2f} pips<extra></extra>"
                ),
                row=row_gap, col=1
            )
            trace_map.append((d, broker, "gap", trace_idx)); trace_idx += 1

            # --- zero line
            fig.add_trace(
                go.Scattergl(
                    x=block_plot["Date"] if len(block_plot) else [],
                    y=np.zeros(len(block_plot)) if len(block_plot) else [],
                    mode="lines",
                    line=dict(color="rgba(180,180,180,0.45)", width=1.0, dash="dash"),
                    showlegend=False,
                    visible=(d == dates[0]),
                    hoverinfo="skip"
                ),
                row=row_gap, col=1
            )
            trace_map.append((d, broker, "gap_zero", trace_idx)); trace_idx += 1

            # --- heavy peak markers (only peaks inside day)
            fig.add_trace(
                go.Scattergl(
                    x=heavy_in_day["peak_time"] if len(heavy_in_day) else [],
                    y=heavy_in_day["peak_gap_pips"] if len(heavy_in_day) else [],
                    mode="markers",
                    marker=dict(color="#ffb454", size=7, symbol="diamond"),
                    showlegend=False,
                    visible=(d == dates[0]),
                    hovertemplate="Time=%{x}<br>Episode peak gap=%{y:.2f} pips<extra></extra>"
                ),
                row=row_gap, col=1
            )
            trace_map.append((d, broker, "peak_gap", trace_idx)); trace_idx += 1

            # --- shading (clip to day)
            if len(ep_day) > 0 and len(block) > 0:
                for _, erow in ep_day.iterrows():
                    ps = max(pd.to_datetime(erow["start_time"]), day_start)
                    pe = min(pd.to_datetime(erow["end_time"]), day_end)
                    if pe <= ps:
                        continue

                    peak_gap = float(erow.get("peak_gap_pips", np.nan))
                    if not np.isfinite(peak_gap):
                        continue

                    is_heavy = bool(erow.get("is_heavy", False))

                    shapes_for_day.extend(
                        build_episode_segment_shapes_clipped(
                            block_df=block,
                            plot_start=ps,
                            plot_end=pe,
                            peak_gap_pips=peak_gap,
                            is_heavy=is_heavy,
                            row_num=row_band,
                            n_segments=n_episode_segments,
                            min_alpha=0.04,
                            max_alpha=0.22,
                            heavy_extra_alpha=0.06
                        )
                    )
                    shapes_for_day.extend(
                        build_episode_segment_shapes_clipped(
                            block_df=block,
                            plot_start=ps,
                            plot_end=pe,
                            peak_gap_pips=peak_gap,
                            is_heavy=is_heavy,
                            row_num=row_gap,
                            n_segments=n_episode_segments,
                            min_alpha=0.05,
                            max_alpha=0.28,
                            heavy_extra_alpha=0.08
                        )
                    )

        layout_shapes_map[d] = shapes_for_day

    fig.update_layout(shapes=layout_shapes_map[dates[0]])

    total_traces = len(fig.data)
    buttons = []
    for d in dates:
        visible = [False] * total_traces
        for dd, broker, tname, idx in trace_map:
            if dd == d:
                visible[idx] = True

        buttons.append(dict(
            label=str(d),
            method="update",
            args=[{"visible": visible}, {"title": f"Three-LP Gap Dashboard — {d}", "shapes": layout_shapes_map[d]}]
        ))

    fig.update_layout(
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            x=1.02, y=1.0,
            xanchor="left", yanchor="top",
            showactive=True
        )],
        template="plotly_dark",
        height=1500,
        title=f"Three-LP Gap Dashboard — {dates[0]}",
        hovermode="x unified",
        paper_bgcolor="#0f1117",
        plot_bgcolor="#151924",
        margin=dict(l=40, r=180, t=80, b=40)
    )

    for i, broker in enumerate(brokers):
        fig.update_yaxes(title_text="Centered band (pips)", row=2*i+1, col=1)
        fig.update_yaxes(title_text="state_gap_pips", row=2*i+2, col=1)

    fig.update_xaxes(title_text="Time", row=2*len(brokers), col=1)
    return fig

In [ ]:
fig = build_three_lp_gap_dashboard_clean_fixed(
    processed_all=processed_viz_all,
    episodes_all=episodes_viz_all,
    max_points_per_broker_day=1400,
    max_events_per_broker_day=200,
    n_episode_segments=8
)
fig.show()

Indeed, as we can see:

\
***Liquidity Provider #1*** has ***the biggest Spread Extentions at US sessions*** but at the same time has ***the smallest baseline spread level***.

\
***Liquidity Provider #2*** has quite ***big extentions during the first hour after Gold session starting*** that can cause ***serious problems for the clients and should be eliminated***.

\
***The Liquidity Provider #3*** has ***the most stable Spread*** but its ***baseline is still quite high***.

## Empirical Patterns of Spread Expansions

The visualizations reveal clear structural differences in how spread expansions occur across liquidity providers and across days of the week.

\
Several key observations emerge.

\
First, spread expansions are not uniformly distributed in time. Instead, they tend to cluster in specific sessions and days, indicating that the underlying spread process is not only mean-reverting but also regime-dependent. This is consistent with the model structure, where the equilibrium level $ \mu_{s_t} $ and volatility scale $ \omega_{s_t} $ depend explicitly on the trading session.

\
Second, the duration of ***spread expansions*** varies significantly across providers. In terms of the latent model, this corresponds to differences in the mean-reversion speed $ \kappa $, since the expected decay of deviations is governed by

\
$$
\mathbb{E}[X_{t+\tau} - \mu \mid X_t] = e^{-\kappa \tau}(X_t - \mu).
$$

\
Providers with smaller $ \kappa $ exhibit longer-lived spread expansions, which is directly visible in prolonged elevated regions in the plots.

\
Third, the amplitude of spread expansions is closely related to both the equilibrium level $ \mu_s $ and the stationary standard deviation $ \omega_s $. Larger values of $ \omega_s $ imply that the ***latent spread process*** has a higher probability of generating extreme deviations, while larger $ \mu_s $ shifts the entire distribution upward.

\
Finally, the observed spread $ Y_t $ often exhibits short-lived spikes that are not persistent. These correspond to the observation-noise component

\
$$
\varepsilon_t \sim \mathcal{N}(0, R_{s_t}),
$$

\
and are especially pronounced for providers with large $ R_s $. Thus, the visualization also reflects the decomposition

\
$$
Y_t = X_t + \varepsilon_t,
$$

\
where persistent expansions are driven by $ X_t $, and high-frequency fluctuations are driven by $ \varepsilon_t $.

Overall, the graphical analysis confirms that ***spread risk*** is a multi-dimensional phenomenon involving amplitude, persistence, and frequency, all of which are captured by the ***estimated state-space model***.

### **Conclusion and Transition to Part II**

In this notebook, we have developed and applied a ***fully specified stochastic framework*** for evaluating liquidity provider quality through the ***behavior of the bid-ask spread***.

***The spread*** was modeled as a ***latent mean-reverting process*** governed by an ***Ornstein-Uhlenbeck Stochastic Process***, observed through noisy quotes and estimated using ***the Kalman filter***. This allowed us to decompose the observed spread dynamics into three fundamental components:

- the equilibrium spread level $ \mu_s $,
- the persistence of deviations governed by $ \kappa $ (or equivalently the half-life $ h $),
- the separation between latent spread variability $ \omega_s $ and observation noise $ R_s $.

Using this framework, we were able to **identify, quantify, and compare spread expansion phenomena** across three liquidity providers. In particular, we have:

- detected episodes of spread widening,
- measured their amplitude, duration, and frequency,
- analyzed their dependence on trading sessions and calendar structure,
- and linked these empirical observations to the underlying stochastic model.

Thus, we have formally constructed and measured the notion of **spread expansion as a stochastic object**, rather than treating it as an ad hoc feature of the data.

---

## Outlook: Toward Feed Latency and Price Gap Risk (Part II)

While spread behavior is a fundamental component of execution quality, it is ***not the only source of risk*** for clients. Even a provider with acceptable spread characteristics may expose clients to significant risk if its quote feed is delayed, irregular, or clustered in time.

In the next part, we will extend the analysis to ***the temporal structure of quote arrivals***. Specifically, we will study:

- the inter-arrival times between successive quotes;
- clustering of delays;
- duration of inactivity periods;
- and the relationship between delayed quotes and the formation of price gaps.

From the perspective of stochastic-process theory, this requires modeling the arrival ***times of quotes as a point process***. Instead of modeling the spread level $ X_t $, we will model the sequence of arrival times $ \{T_i\} $ and durations
$
\Delta_i = T_i - T_{i-1}.
$

\
To capture clustering and volatility of durations, we will use an ACD-ECOGARCH-type framework, where:

- the ACD ***(Autoregressive Conditional Duration)*** component models the conditional expectation of durations,
- the ECOGARCH component models the logarithmic volatility of durations in a manner analogous to ***continuous-time stochastic volatility models***.

This approach can be seen as a ***discrete-time analogue of stochastic intensity models for point processes***, where the conditional intensity itself evolves randomly over time.

In combination with the spread model developed in this notebook, this will allow us to link **quote timing**, **spread behavior**, and **price-gap formation** into a unified framework for ***assessing execution risk and client safety***.

---

**Thank you for your attention.**

In [ ]:
import json

notebook_path = "LP quality estimation - Part 1 Spread Widening detection.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "metadata" in nb and "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Removed metadata.widgets")

In [ ]:
import json
import os

notebook_name = "LP quality estimation - Part 1 Spread Widening detection.ipynb"

from google.colab import _message
nb = _message.blocking_request("get_ipynb")["ipynb"]

# удаляем widgets metadata
if "metadata" in nb and "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

# чистим outputs
for cell in nb.get("cells", []):
    if "outputs" in cell:
        cell["outputs"] = []
    if "execution_count" in cell:
        cell["execution_count"] = None

# путь сохранения (текущая директория Colab)
save_path = os.path.abspath(notebook_name)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("✅ Notebook saved at:")
print(save_path)